# Colab patch notebook

Run these cells from `/content` after cloning/uploading `smart-flip`. They overwrite the project files needed for the remaining 3-bit +BC runs.


In [ ]:
%%bash
mkdir -p smart-flip/flatquant/model_tools
rm -f smart-flip/flatquant/model_tools/llama_utils.py


In [ ]:
%%writefile smart-flip/flatquant/model_tools/llama_utils.py
import inspect
import math

import torch
import torch.nn as nn

from flatquant.quant_utils import ActivationQuantizer
from flatquant.utils import skip_initialization
from flatquant.function_utils import get_init_scale, get_decompose_dim
from flatquant.trans_utils import SVDSingleTransMatrix, SVDDecomposeTransMatrix
from flatquant.trans_utils import InvSingleTransMatrix, InvDecomposeTransMatrix
from flatquant.flat_linear import FlatQuantizedLinear

from transformers.models.llama.modeling_llama import LlamaMLP, LlamaAttention, LlamaRotaryEmbedding, \
                                                     apply_rotary_pos_emb, repeat_kv


from tqdm import tqdm

class FlatQuantLlamaMLP(LlamaMLP):
    def __init__(self, args, module: LlamaMLP):
        super().__init__(module.config)
        self.args = args
        self.up_proj = FlatQuantizedLinear(args, module.up_proj)
        self.gate_proj = FlatQuantizedLinear(args, module.gate_proj)
        self.down_proj = FlatQuantizedLinear(args, module.down_proj)
        self.add_fq_trans()

        self._ori_mode = False
        self.diag_init = args.diag_init
        if self.diag_init == "sq_style":
            self.up_smax = torch.ones_like(self.up_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5
            self.down_smax = torch.ones_like(self.down_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5
        
    def add_fq_trans(self):
        if self.args.direct_inv:
            DecomposeTransMatrix = InvDecomposeTransMatrix
        else:
            DecomposeTransMatrix = SVDDecomposeTransMatrix
        if self.args.w_bits < 16 or self.args.a_bits < 16:
            up_dim_left, up_dim_right = get_decompose_dim(self.up_proj.linear.weight.shape[1])
            self.up_gate_trans = DecomposeTransMatrix(up_dim_left, up_dim_right, add_diag=self.args.add_diag)
            down_dim_left, down_dim_right = get_decompose_dim(self.down_proj.linear.weight.shape[1])
            self.down_trans = DecomposeTransMatrix(down_dim_left, down_dim_right, add_diag=self.args.add_diag)
        else:
            self.up_gate_trans, self.down_trans = None, None

    def _trans_forward(self, x):
        if self.up_gate_trans is not None:
            x_ts = self.up_gate_trans(x)
        else:
            x_ts = x
        up_states = self.up_proj(x_ts, qa_trans=self.up_gate_trans)
        gate_states = self.gate_proj(x_ts, qa_trans=self.up_gate_trans)

        x_act_fn = self.act_fn(gate_states) * up_states
        if self.down_trans is not None:
            x_ts_2 = self.down_trans(x_act_fn)
        else:
            x_ts_2 = x_act_fn
        down_states = self.down_proj(x_ts_2, qa_trans=self.down_trans)
        return down_states

    def _ori_forward(self, x):
        '''origin implement: down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))'''
        if self.diag_init == "sq_style":
            self.up_smax = torch.maximum(self.up_smax, x.reshape(-1, x.shape[-1]).abs().max(0)[0].clone().detach())
        x = self.act_fn(self.gate_proj._ori_forward(x)) * self.up_proj._ori_forward(x)
        if self.diag_init == "sq_style":
            self.down_smax = torch.maximum(self.down_smax, x.reshape(-1, x.shape[-1]).abs().max(0)[0].clone().detach())
        down_states = self.down_proj._ori_forward(x)
        return down_states

    def forward(self, x):
        if self._ori_mode:
            return self._ori_forward(x)
        return self._trans_forward(x)

    def reparameterize(self, ):
        if self.up_gate_trans is not None:
            self.up_gate_trans.to_eval_mode()
            self.down_trans.to_eval_mode()
        self.gate_proj.reparameterize(qa_trans=self.up_gate_trans)
        self.up_proj.reparameterize(qa_trans=self.up_gate_trans)
        self.down_proj.reparameterize(qa_trans=self.down_trans)
        if self.up_gate_trans is not None:
            self.up_gate_trans.use_diag = False
        # merge trans's diag scale
        if self.down_trans is not None and self.down_trans.add_diag:
            up_weight = self.up_proj.linear.weight
            ori_dtype = up_weight.dtype
            up_weight = up_weight.to(torch.float64).T.mul(self.down_trans.diag_scale.to(torch.float64)).T
            self.up_proj.linear.weight.data = up_weight.to(ori_dtype)
            self.down_trans.use_diag = False

    def init_diag_scale(self, alpha=0.5):
        assert hasattr(self, "up_smax") and hasattr(self, "down_smax")
        upw_smax = torch.cat([self.up_proj.linear.weight, self.gate_proj.linear.weight], dim=0).abs().max(dim=0)[0]
        downw_smax = self.down_proj.linear.weight.abs().max(dim=0)[0]
        if self.up_gate_trans is not None:
            self.up_gate_trans.diag_scale.data = get_init_scale(upw_smax, self.up_smax, alpha)
        if self.down_trans is not None:
            self.down_trans.diag_scale.data = get_init_scale(downw_smax, self.down_smax, alpha)
        del self.up_smax, self.down_smax
        self.diag_init = None

    def rep_matrix_only(self, ):
        if self.up_gate_trans is not None:
            self.up_gate_trans.to_eval_mode()
            self.down_trans.to_eval_mode()


class FlatQuantLlamaAttention(LlamaAttention):
    @staticmethod
    def _build_rotary_embedding(module):
        config = module.config
        device = module.q_proj.weight.device
        attempts = [
            lambda: LlamaRotaryEmbedding(config=config, device=device),
            lambda: LlamaRotaryEmbedding(config=config),
            lambda: LlamaRotaryEmbedding(config, device=device),
            lambda: LlamaRotaryEmbedding(config),
        ]
        last_error = None
        for attempt in attempts:
            try:
                return attempt()
            except (TypeError, AttributeError) as exc:
                last_error = exc
        if last_error is not None:
            raise last_error
        raise RuntimeError("Unable to construct LlamaRotaryEmbedding")

    def _compute_position_embeddings(self, value_states, position_ids):
        if not hasattr(self, "rotary_emb"):
            raise AttributeError("FlatQuantLlamaAttention has no rotary_emb fallback")

        attempts = [
            lambda: self.rotary_emb(value_states, position_ids),
            lambda: self.rotary_emb(value_states, position_ids=position_ids),
            lambda: self.rotary_emb(position_ids=position_ids, x=value_states),
        ]
        last_error = None
        for attempt in attempts:
            try:
                return attempt()
            except TypeError as exc:
                last_error = exc
        if last_error is not None:
            raise last_error
        raise RuntimeError("Unable to compute Llama rotary position embeddings")

    def __init__(self, args, module: LlamaAttention):
        super().__init__(module.config, module.layer_idx)
        self.args = args
        self._use_position_embeddings_api = "position_embeddings" in inspect.signature(module.forward).parameters
        self.hidden_size = getattr(module, "hidden_size", module.config.hidden_size)
        self.num_heads = getattr(module, "num_heads", getattr(module, "num_attention_heads", module.config.num_attention_heads))
        self.num_key_value_heads = getattr(module, "num_key_value_heads", module.config.num_key_value_heads)
        self.head_dim = getattr(module, "head_dim", getattr(module.config, "head_dim", self.hidden_size // self.num_heads))
        self.num_key_value_groups = getattr(module, "num_key_value_groups", self.num_heads // self.num_key_value_heads)
        self.scaling = getattr(module, "scaling", self.head_dim ** -0.5)
        self.attention_dropout = getattr(module, "attention_dropout", getattr(module.config, "attention_dropout", 0.0))
        self.is_causal = getattr(module, "is_causal", True)
        if hasattr(module, "rotary_emb"):
            self.rotary_emb = module.rotary_emb
        else:
            self.rotary_emb = self._build_rotary_embedding(module)
        
        self.q_proj = FlatQuantizedLinear(args, module.q_proj)
        self.k_proj = FlatQuantizedLinear(args, module.k_proj)
        self.v_proj = FlatQuantizedLinear(args, module.v_proj)
        self.o_proj = FlatQuantizedLinear(args, module.o_proj)
        self.add_fq_trans()

        if args.q_bits < 16:
            self.q_cache_quantizer = ActivationQuantizer(bits=args.q_bits, \
                                        sym=not(args.q_asym), lac=args.lac, groupsize=-1, )
        if args.k_bits < 16:
            self.k_cache_quantizer = ActivationQuantizer(bits=args.k_bits, \
                                        sym=not(args.k_asym), lac=args.lac, groupsize=-1, )
        if args.v_bits < 16:
            self.v_cache_quantizer = ActivationQuantizer(bits=args.v_bits, \
                                        sym=not(args.v_asym), lac=args.lac, groupsize=-1, )

        self._ori_mode = False
        self._eval_mode = False
        self.diag_init = args.diag_init
        if self.diag_init == "sq_style":
            self.ln_smax = torch.ones_like(self.q_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5

    def add_fq_trans(self):
        if self.args.direct_inv:
            SingleTransMatrix, DecomposeTransMatrix = InvSingleTransMatrix, InvDecomposeTransMatrix
        else:
            SingleTransMatrix, DecomposeTransMatrix = SVDSingleTransMatrix, SVDDecomposeTransMatrix
        if self.args.w_bits < 16 or self.args.a_bits < 16:
            ln_dim_left, ln_dim_right = get_decompose_dim(self.q_proj.linear.weight.shape[1])
            self.ln_trans = DecomposeTransMatrix(ln_dim_left, ln_dim_right, add_diag=self.args.add_diag)
            self.o_trans = SingleTransMatrix(self.config.num_attention_heads)
        else:
            self.ln_trans, self.o_trans = None, None

        head_dim = self.config.hidden_size // self.config.num_attention_heads
        if self.args.k_bits < 16 or self.args.q_bits < 16:
            self.kcache_trans = SingleTransMatrix(head_dim)
        else:
            self.kcache_trans = None
        if self.args.v_bits < 16 or self.args.w_bits < 16 or self.args.a_bits < 16:
            self.vcache_trans = SingleTransMatrix(head_dim)
        else:
            self.vcache_trans = None

    def _trans_forward_after_ln(self, hidden_states):
        if self.ln_trans is not None:
            hidden_states = self.ln_trans(hidden_states)
        query_states = self.q_proj(hidden_states, qa_trans=self.ln_trans)
        key_states = self.k_proj(hidden_states, qa_trans=self.ln_trans)
        if self.args.separate_vtrans:
            value_states = self.v_proj(hidden_states, qa_trans=self.ln_trans)
        else:
            value_states = self.v_proj(hidden_states, qa_trans=self.ln_trans, out_trans=self.vcache_trans)
        return query_states, key_states, value_states

    def _ori_forward_after_ln(self, hidden_states):
        if self.diag_init == "sq_style" and hasattr(self, "ln_smax"):
            self.ln_smax = torch.maximum(self.ln_smax, \
                hidden_states.reshape(-1, hidden_states.shape[-1]).abs().max(0)[0].clone().detach())
        query_states = self.q_proj._ori_forward(hidden_states)
        key_states = self.k_proj._ori_forward(hidden_states)
        value_states = self.v_proj._ori_forward(hidden_states)
        return query_states, key_states, value_states

    def quant_vcache(self, value_states):
        if self.args.separate_vtrans:
            value_states = self.vcache_trans(value_states)
        if self.args.v_bits < 16:
            value_states = self.v_cache_quantizer(value_states)
        return value_states

    def quant_kcache(self, q, k):
        if not (self.args.k_bits < 16 or self.args.q_bits < 16):
            return q, k
        # Q/K transform
        if self.kcache_trans is not None:
            q = self.kcache_trans(q, inv_t=True)
            k = self.kcache_trans(k)
        if self.args.q_bits < 16:
            q = self.q_cache_quantizer(q).to(q)
        # TODO: by default do the per-head quantizaion for k-v-cache
        if self.args.k_bits < 16:
            k = self.k_cache_quantizer(k).to(q)
        return q, k

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,
        past_key_values=None,
        **kwargs,
    ):
        cache_obj = past_key_values if past_key_values is not None else past_key_value
        bsz, q_len, _ = hidden_states.size()
        if self._ori_mode:
            query_states, key_states, value_states = self._ori_forward_after_ln(hidden_states)
        else:
            query_states, key_states, value_states = self._trans_forward_after_ln(hidden_states)

        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        if position_embeddings is None:
            cos, sin = self._compute_position_embeddings(value_states, position_ids)
        else:
            cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
        # ---- here do the quantization ----
        if not self._ori_mode:
            query_states, key_states = self.quant_kcache(query_states, key_states)
            value_states = self.quant_vcache(value_states)

        if cache_obj is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            key_states, value_states = cache_obj.update(key_states, value_states, self.layer_idx, cache_kwargs)

        key_states = repeat_kv(key_states, self.num_key_value_groups)
        value_states = repeat_kv(value_states, self.num_key_value_groups) # bnsh
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(self.head_dim)

        if attention_mask is not None:
            causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]
            attn_weights = attn_weights + causal_mask

        # upcast attention to fp32
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_weights = nn.functional.dropout(attn_weights, p=self.attention_dropout, training=self.training)
        attn_output = torch.matmul(attn_weights, value_states)

        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):
            raise ValueError(
                f"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is"
                f" {attn_output.size()}"
            )

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(bsz, q_len, -1)
        if self._ori_mode:
            attn_output = self.o_proj._ori_forward(attn_output)
        else:
            # new foward: 
            if self.o_trans is None and self.vcache_trans is not None:
                # attn_output = self.vcache_trans(value_states)
                init_shape = attn_output.shape
                attn_output = attn_output.reshape(-1, self.config.num_attention_heads, self.config.hidden_size//self.config.num_attention_heads)
                attn_output = torch.matmul(attn_output, self.vcache_trans.get_matrix(inv_t=True).T.to(attn_output)).reshape(init_shape)
                attn_output = self.o_proj(attn_output)
            else:
                init_shape = attn_output.shape
                attn_output = attn_output.reshape(-1, self.config.num_attention_heads, self.config.hidden_size//self.config.num_attention_heads)
                attn_output = torch.matmul(self.o_trans.get_matrix().T.to(attn_output), attn_output).reshape(init_shape)
                if not self._eval_mode:
                    attn_o_og_it = self.o_trans.get_matrix(inv_t=True)
                    attn_v_og_it = self.vcache_trans.get_matrix(inv_t=True)
                    attn_output = self.o_proj(attn_output, qa_trans=[attn_o_og_it, attn_v_og_it])
                else:
                    attn_output = self.o_proj(attn_output)
        if not output_attentions:
            attn_weights = None
        return attn_output, attn_weights, cache_obj

    def reparameterize(self):
        if self.ln_trans is not None:
            self.ln_trans.to_eval_mode()
        if self.kcache_trans is not None:
            self.kcache_trans.to_eval_mode()
        if self.vcache_trans is not None:
            self.vcache_trans.to_eval_mode()
        if self.o_trans is not None:
            self.o_trans.to_eval_mode()
        self.q_proj.reparameterize(qa_trans=self.ln_trans)
        self.k_proj.reparameterize(qa_trans=self.ln_trans)
        if self.args.separate_vtrans:
            self.v_proj.reparameterize(qa_trans=self.ln_trans)
        else:
            self.v_proj.reparameterize(qa_trans=self.ln_trans, out_trans=self.vcache_trans)
        if self.o_trans is not None and self.vcache_trans is not None:
            attn_o_og_it = self.o_trans.get_matrix(inv_t=True)
            attn_v_og_it = self.vcache_trans.get_matrix(inv_t=True)
            self.o_proj.reparameterize(qa_trans=[attn_o_og_it, attn_v_og_it])
        self._eval_mode = True

    def init_diag_scale(self, alpha=0.5):
        assert hasattr(self, "ln_smax")
        qkvw_smax = torch.cat([self.q_proj.linear.weight, self.k_proj.linear.weight, self.v_proj.linear.weight], dim=0).abs().max(dim=0)[0]
        if self.ln_trans is not None:
            self.ln_trans.diag_scale.data = get_init_scale(qkvw_smax, self.ln_smax, alpha)
        del self.ln_smax
        self.diag_init = None

    def rep_matrix_only(self, ):
        if self.ln_trans is not None:
            self.ln_trans.to_eval_mode()
        if self.kcache_trans is not None:
            self.kcache_trans.to_eval_mode()
        if self.vcache_trans is not None:
            self.vcache_trans.to_eval_mode()
        if self.o_trans is not None:
            self.o_trans.to_eval_mode()


def apply_flatquant_to_llama(args, model):
    skip_initialization()
    # Replace module with FlatQuant version
    for layer in tqdm(range(model.config.num_hidden_layers), desc="Applying FlatQuant to model"):
        # attn
        model.model.layers[layer].self_attn = FlatQuantLlamaAttention(args, model.model.layers[layer].self_attn)
        # mlp
        model.model.layers[layer].mlp = FlatQuantLlamaMLP(args, model.model.layers[layer].mlp)
    return model


In [ ]:
%%bash
mkdir -p smart-flip/flatquant/model_tools
rm -f smart-flip/flatquant/model_tools/llama31_utils.py


In [ ]:
%%writefile smart-flip/flatquant/model_tools/llama31_utils.py
import inspect
import math

import torch
import torch.nn as nn

from flatquant.quant_utils import ActivationQuantizer
from flatquant.utils import skip_initialization
from flatquant.function_utils import get_init_scale, get_decompose_dim
from flatquant.trans_utils import SVDSingleTransMatrix, SVDDecomposeTransMatrix
from flatquant.trans_utils import InvSingleTransMatrix, InvDecomposeTransMatrix
from flatquant.flat_linear import FlatQuantizedLinear

from transformers.models.llama.modeling_llama import LlamaMLP, LlamaAttention, LlamaRotaryEmbedding, \
                                                     apply_rotary_pos_emb, repeat_kv


class FlatQuantLlamaMLP(LlamaMLP):
    def __init__(self, args, module: LlamaMLP):
        super().__init__(module.config)
        self.args = args
        self.up_proj = FlatQuantizedLinear(args, module.up_proj)
        self.gate_proj = FlatQuantizedLinear(args, module.gate_proj)
        self.down_proj = FlatQuantizedLinear(args, module.down_proj)
        self.add_fq_trans()

        self._ori_mode = False
        self.diag_init = args.diag_init
        if self.diag_init == "sq_style":
            self.up_smax = torch.ones_like(self.up_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5
            self.down_smax = torch.ones_like(self.down_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5
        
    def add_fq_trans(self):
        if self.args.direct_inv:
            DecomposeTransMatrix = InvDecomposeTransMatrix
        else:
            DecomposeTransMatrix = SVDDecomposeTransMatrix
        if self.args.w_bits < 16 or self.args.a_bits < 16:
            up_dim_left, up_dim_right = get_decompose_dim(self.up_proj.linear.weight.shape[1])
            self.up_gate_trans = DecomposeTransMatrix(up_dim_left, up_dim_right, add_diag=self.args.add_diag)
            down_dim_left, down_dim_right = get_decompose_dim(self.down_proj.linear.weight.shape[1])
            self.down_trans = DecomposeTransMatrix(down_dim_left, down_dim_right, add_diag=self.args.add_diag)
        else:
            self.up_gate_trans, self.down_trans = None, None

    def _trans_forward(self, x):
        if self.up_gate_trans is not None:
            x_ts = self.up_gate_trans(x)
        else:
            x_ts = x
        up_states = self.up_proj(x_ts, qa_trans=self.up_gate_trans)
        gate_states = self.gate_proj(x_ts, qa_trans=self.up_gate_trans)

        x_act_fn = self.act_fn(gate_states) * up_states
        if self.down_trans is not None:
            x_ts_2 = self.down_trans(x_act_fn)
        else:
            x_ts_2 = x_act_fn
        down_states = self.down_proj(x_ts_2, qa_trans=self.down_trans)
        return down_states

    def _ori_forward(self, x):
        '''origin implement: down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))'''
        if self.diag_init == "sq_style":
            self.up_smax = torch.maximum(self.up_smax, x.reshape(-1, x.shape[-1]).abs().max(0)[0].clone().detach())
        x = self.act_fn(self.gate_proj._ori_forward(x)) * self.up_proj._ori_forward(x)
        if self.diag_init == "sq_style":
            self.down_smax = torch.maximum(self.down_smax, x.reshape(-1, x.shape[-1]).abs().max(0)[0].clone().detach())
        down_states = self.down_proj._ori_forward(x)
        return down_states

    def forward(self, x):
        if self._ori_mode:
            return self._ori_forward(x)
        return self._trans_forward(x)

    def reparameterize(self, ):
        if self.up_gate_trans is not None:
            self.up_gate_trans.to_eval_mode()
            self.down_trans.to_eval_mode()
        self.gate_proj.reparameterize(qa_trans=self.up_gate_trans)
        self.up_proj.reparameterize(qa_trans=self.up_gate_trans)
        self.down_proj.reparameterize(qa_trans=self.down_trans)
        if self.up_gate_trans is not None:
            self.up_gate_trans.use_diag = False
        # merge trans's diag scale
        if self.down_trans is not None and self.down_trans.add_diag:
            up_weight = self.up_proj.linear.weight
            ori_dtype = up_weight.dtype
            up_weight = up_weight.to(torch.float64).T.mul(self.down_trans.diag_scale.to(torch.float64)).T
            self.up_proj.linear.weight.data = up_weight.to(ori_dtype)
            self.down_trans.use_diag = False

    def init_diag_scale(self, alpha=0.5):
        assert hasattr(self, "up_smax") and hasattr(self, "down_smax")
        upw_smax = torch.cat([self.up_proj.linear.weight, self.gate_proj.linear.weight], dim=0).abs().max(dim=0)[0]
        downw_smax = self.down_proj.linear.weight.abs().max(dim=0)[0]
        if self.up_gate_trans is not None:
            self.up_gate_trans.diag_scale.data = get_init_scale(upw_smax, self.up_smax, alpha)
        if self.down_trans is not None:
            self.down_trans.diag_scale.data = get_init_scale(downw_smax, self.down_smax, alpha)
        del self.up_smax, self.down_smax
        self.diag_init = None

    def rep_matrix_only(self, ):
        if self.up_gate_trans is not None:
            self.up_gate_trans.to_eval_mode()
            self.down_trans.to_eval_mode()


class FlatQuantLlamaAttention(LlamaAttention):
    @staticmethod
    def _build_rotary_embedding(module):
        config = module.config
        device = module.q_proj.weight.device
        attempts = [
            lambda: LlamaRotaryEmbedding(config=config, device=device),
            lambda: LlamaRotaryEmbedding(config=config),
            lambda: LlamaRotaryEmbedding(config, device=device),
            lambda: LlamaRotaryEmbedding(config),
        ]
        last_error = None
        for attempt in attempts:
            try:
                return attempt()
            except (TypeError, AttributeError) as exc:
                last_error = exc
        if last_error is not None:
            raise last_error
        raise RuntimeError("Unable to construct LlamaRotaryEmbedding")

    def _compute_position_embeddings(self, value_states, position_ids):
        if not hasattr(self, "rotary_emb"):
            raise AttributeError("FlatQuantLlamaAttention has no rotary_emb fallback")

        attempts = [
            lambda: self.rotary_emb(value_states, position_ids),
            lambda: self.rotary_emb(value_states, position_ids=position_ids),
            lambda: self.rotary_emb(position_ids=position_ids, x=value_states),
        ]
        last_error = None
        for attempt in attempts:
            try:
                return attempt()
            except TypeError as exc:
                last_error = exc
        if last_error is not None:
            raise last_error
        raise RuntimeError("Unable to compute Llama rotary position embeddings")

    def __init__(self, args, module: LlamaAttention):
        super().__init__(module.config, module.layer_idx)
        self.args = args
        self._use_position_embeddings_api = "position_embeddings" in inspect.signature(module.forward).parameters
        self.hidden_size = getattr(module, "hidden_size", module.config.hidden_size)
        self.num_heads = getattr(module, "num_heads", getattr(module, "num_attention_heads", module.config.num_attention_heads))
        self.num_key_value_heads = getattr(module, "num_key_value_heads", module.config.num_key_value_heads)
        self.head_dim = getattr(module, "head_dim", getattr(module.config, "head_dim", self.hidden_size // self.num_heads))
        self.num_key_value_groups = getattr(module, "num_key_value_groups", self.num_heads // self.num_key_value_heads)
        self.scaling = getattr(module, "scaling", self.head_dim ** -0.5)
        self.attention_dropout = getattr(module, "attention_dropout", getattr(module.config, "attention_dropout", 0.0))
        self.is_causal = getattr(module, "is_causal", True)
        if hasattr(module, "rotary_emb"):
            self.rotary_emb = module.rotary_emb
        else:
            self.rotary_emb = self._build_rotary_embedding(module)
        
        self.q_proj = FlatQuantizedLinear(args, module.q_proj)
        self.k_proj = FlatQuantizedLinear(args, module.k_proj)
        self.v_proj = FlatQuantizedLinear(args, module.v_proj)
        self.o_proj = FlatQuantizedLinear(args, module.o_proj)
        self.add_fq_trans()

        if args.q_bits < 16:
            self.q_cache_quantizer = ActivationQuantizer(bits=args.q_bits, \
                                        sym=not(args.q_asym), lac=args.lac, groupsize=-1, )
        if args.k_bits < 16:
            self.k_cache_quantizer = ActivationQuantizer(bits=args.k_bits, \
                                        sym=not(args.k_asym), lac=args.lac, groupsize=-1, )
        if args.v_bits < 16:
            self.v_cache_quantizer = ActivationQuantizer(bits=args.v_bits, \
                                        sym=not(args.v_asym), lac=args.lac, groupsize=-1, )

        self._ori_mode = False
        self._eval_mode = False
        self.diag_init = args.diag_init
        if self.diag_init == "sq_style":
            self.ln_smax = torch.ones_like(self.q_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5

    def add_fq_trans(self):
        if self.args.direct_inv:
            SingleTransMatrix, DecomposeTransMatrix = InvSingleTransMatrix, InvDecomposeTransMatrix
        else:
            SingleTransMatrix, DecomposeTransMatrix = SVDSingleTransMatrix, SVDDecomposeTransMatrix
        if self.args.w_bits < 16 or self.args.a_bits < 16:
            ln_dim_left, ln_dim_right = get_decompose_dim(self.q_proj.linear.weight.shape[1])
            self.ln_trans = DecomposeTransMatrix(ln_dim_left, ln_dim_right, add_diag=self.args.add_diag)
            self.o_trans = SingleTransMatrix(self.config.num_attention_heads)
        else:
            self.ln_trans, self.o_trans = None, None

        head_dim = self.config.hidden_size // self.config.num_attention_heads
        if self.args.k_bits < 16 or self.args.q_bits < 16:
            self.kcache_trans = SingleTransMatrix(head_dim)
        else:
            self.kcache_trans = None
        if self.args.v_bits < 16 or self.args.w_bits < 16 or self.args.a_bits < 16:
            self.vcache_trans = SingleTransMatrix(head_dim)
        else:
            self.vcache_trans = None

    def _trans_forward_after_ln(self, hidden_states):
        if self.ln_trans is not None:
            hidden_states = self.ln_trans(hidden_states)
        query_states = self.q_proj(hidden_states, qa_trans=self.ln_trans)
        key_states = self.k_proj(hidden_states, qa_trans=self.ln_trans)
        if self.args.separate_vtrans:
            value_states = self.v_proj(hidden_states, qa_trans=self.ln_trans)
        else:
            value_states = self.v_proj(hidden_states, qa_trans=self.ln_trans, out_trans=self.vcache_trans)
        return query_states, key_states, value_states

    def _ori_forward_after_ln(self, hidden_states):
        if self.diag_init == "sq_style" and hasattr(self, "ln_smax"):
            self.ln_smax = torch.maximum(self.ln_smax, \
                hidden_states.reshape(-1, hidden_states.shape[-1]).abs().max(0)[0].clone().detach())
        query_states = self.q_proj._ori_forward(hidden_states)
        key_states = self.k_proj._ori_forward(hidden_states)
        value_states = self.v_proj._ori_forward(hidden_states)
        return query_states, key_states, value_states

    def quant_vcache(self, value_states):
        if self.args.separate_vtrans:
            value_states = self.vcache_trans(value_states)
        if self.args.v_bits < 16:
            value_states = self.v_cache_quantizer(value_states)
        return value_states

    def quant_kcache(self, q, k):
        if not (self.args.k_bits < 16 or self.args.q_bits < 16):
            return q, k
        # Q/K transform
        if self.kcache_trans is not None:
            q = self.kcache_trans(q, inv_t=True)
            k = self.kcache_trans(k)
        if self.args.q_bits < 16:
            q = self.q_cache_quantizer(q).to(q)
        # TODO: by default do the per-head quantizaion for k-v-cache
        if self.args.k_bits < 16:
            k = self.k_cache_quantizer(k).to(q)
        return q, k

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,
        past_key_values=None,
        **kwargs,
    ):
        cache_obj = past_key_values if past_key_values is not None else past_key_value
        bsz, q_len, _ = hidden_states.size()
        if self._ori_mode:
            query_states, key_states, value_states = self._ori_forward_after_ln(hidden_states)
        else:
            query_states, key_states, value_states = self._trans_forward_after_ln(hidden_states)

        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        if position_embeddings is None:
            # logger.warning_once(
            #     "The attention layers in this model are transitioning from computing the RoPE embeddings internally "
            #     "through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed "
            #     "`position_embeddings` (Tuple of tensors, containing cos and sin). In v4.45 `position_ids` will be "
            #     "removed and `position_embeddings` will be mandatory."
            # )
            cos, sin = self._compute_position_embeddings(value_states, position_ids)
        else:
            cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
        # ---- here do the quantization ----
        if not self._ori_mode:
            query_states, key_states = self.quant_kcache(query_states, key_states)
            value_states = self.quant_vcache(value_states)

        if cache_obj is not None:
            # sin and cos are specific to RoPE models; cache_position needed for the static cache
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            key_states, value_states = cache_obj.update(key_states, value_states, self.layer_idx, cache_kwargs)

        key_states = repeat_kv(key_states, self.num_key_value_groups)
        value_states = repeat_kv(value_states, self.num_key_value_groups) # bnsh
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(self.head_dim)

        if attention_mask is not None:  # no matter the length, we just slice it
            causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]
            attn_weights = attn_weights + causal_mask

        # upcast attention to fp32
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_weights = nn.functional.dropout(attn_weights, p=self.attention_dropout, training=self.training)
        attn_output = torch.matmul(attn_weights, value_states)

        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):
            raise ValueError(
                f"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is"
                f" {attn_output.size()}"
            )

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(bsz, q_len, -1)
        if self._ori_mode:
            attn_output = self.o_proj._ori_forward(attn_output)
        else:
            # new foward: 
            if self.o_trans is None and self.vcache_trans is not None:
                # attn_output = self.vcache_trans(value_states)
                init_shape = attn_output.shape
                attn_output = attn_output.reshape(-1, self.config.num_attention_heads, self.config.hidden_size//self.config.num_attention_heads)
                attn_output = torch.matmul(attn_output, self.vcache_trans.get_matrix(inv_t=True).T.to(attn_output)).reshape(init_shape)
                attn_output = self.o_proj(attn_output)
            else:
                init_shape = attn_output.shape
                attn_output = attn_output.reshape(-1, self.config.num_attention_heads, self.config.hidden_size//self.config.num_attention_heads)
                attn_output = torch.matmul(self.o_trans.get_matrix().T.to(attn_output), attn_output).reshape(init_shape)
                if not self._eval_mode:
                    attn_o_og_it = self.o_trans.get_matrix(inv_t=True)
                    attn_v_og_it = self.vcache_trans.get_matrix(inv_t=True)
                    attn_output = self.o_proj(attn_output, qa_trans=[attn_o_og_it, attn_v_og_it])
                else:
                    attn_output = self.o_proj(attn_output)

        if not output_attentions:
            attn_weights = None
        return attn_output, attn_weights, cache_obj

    def reparameterize(self):
        if self.ln_trans is not None:
            self.ln_trans.to_eval_mode()
        if self.kcache_trans is not None:
            self.kcache_trans.to_eval_mode()
        if self.vcache_trans is not None:
            self.vcache_trans.to_eval_mode()
        if self.o_trans is not None:
            self.o_trans.to_eval_mode()
        self.q_proj.reparameterize(qa_trans=self.ln_trans)
        self.k_proj.reparameterize(qa_trans=self.ln_trans)
        if self.args.separate_vtrans:
            self.v_proj.reparameterize(qa_trans=self.ln_trans)
        else:
            self.v_proj.reparameterize(qa_trans=self.ln_trans, out_trans=self.vcache_trans)
        if self.o_trans is not None and self.vcache_trans is not None:
            attn_o_og_it = self.o_trans.get_matrix(inv_t=True)
            attn_v_og_it = self.vcache_trans.get_matrix(inv_t=True)
            self.o_proj.reparameterize(qa_trans=[attn_o_og_it, attn_v_og_it])
        self._eval_mode = True

    def init_diag_scale(self, alpha=0.5):
        assert hasattr(self, "ln_smax")
        qkvw_smax = torch.cat([self.q_proj.linear.weight, self.k_proj.linear.weight, self.v_proj.linear.weight], dim=0).abs().max(dim=0)[0]
        if self.ln_trans is not None:
            self.ln_trans.diag_scale.data = get_init_scale(qkvw_smax, self.ln_smax, alpha)
        del self.ln_smax
        self.diag_init = None

    def rep_matrix_only(self, ):
        if self.ln_trans is not None:
            self.ln_trans.to_eval_mode()
        if self.kcache_trans is not None:
            self.kcache_trans.to_eval_mode()
        if self.vcache_trans is not None:
            self.vcache_trans.to_eval_mode()
        if self.o_trans is not None:
            self.o_trans.to_eval_mode()


def apply_flatquant_to_llama_31(args, model):
    skip_initialization()
    # Replace module with FlatQuant version
    for layer in range(model.config.num_hidden_layers):
        # attn
        model.model.layers[layer].self_attn = FlatQuantLlamaAttention(args, model.model.layers[layer].self_attn)
        # mlp
        model.model.layers[layer].mlp = FlatQuantLlamaMLP(args, model.model.layers[layer].mlp)
    return model


In [ ]:
%%bash
mkdir -p smart-flip/flatquant/model_tools
rm -f smart-flip/flatquant/model_tools/qwen_utils.py


In [ ]:
%%writefile smart-flip/flatquant/model_tools/qwen_utils.py
import inspect
import math

import torch
import torch.nn as nn

from flatquant.quant_utils import ActivationQuantizer
from flatquant.utils import skip_initialization
from flatquant.function_utils import get_init_scale, get_decompose_dim
from flatquant.trans_utils import SVDSingleTransMatrix, SVDDecomposeTransMatrix
from flatquant.trans_utils import InvSingleTransMatrix, InvDecomposeTransMatrix
from flatquant.flat_linear import FlatQuantizedLinear

from transformers.models.qwen2.modeling_qwen2 import Qwen2MLP, Qwen2Attention, Qwen2RotaryEmbedding, \
                                                     apply_rotary_pos_emb, repeat_kv



class FlatQuantQwen2MLP(torch.nn.Module):
    def __init__(self, args, module: Qwen2MLP):
        super().__init__()
        self.args = args
        self.hidden_size = module.hidden_size
        self.intermediate_size = module.intermediate_size
        self.act_fn = module.act_fn
        self.up_proj = FlatQuantizedLinear(args, module.up_proj)
        self.gate_proj = FlatQuantizedLinear(args, module.gate_proj)
        self.down_proj = FlatQuantizedLinear(args, module.down_proj)
        self.add_fq_trans()

        self._ori_mode = False
        self.diag_init = args.diag_init
        if self.diag_init == "sq_style":
            self.up_smax = torch.ones_like(self.up_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5
            self.down_smax = torch.ones_like(self.down_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5
        
    def add_fq_trans(self):
        if self.args.direct_inv:
            DecomposeTransMatrix = InvDecomposeTransMatrix
        else:
            DecomposeTransMatrix = SVDDecomposeTransMatrix
        if self.args.w_bits < 16 or self.args.a_bits < 16:
            up_dim_left, up_dim_right = get_decompose_dim(self.up_proj.linear.weight.shape[1])
            self.up_gate_trans = DecomposeTransMatrix(up_dim_left, up_dim_right, add_diag=self.args.add_diag)
            down_dim_left, down_dim_right = get_decompose_dim(self.down_proj.linear.weight.shape[1])
            self.down_trans = DecomposeTransMatrix(down_dim_left, down_dim_right, add_diag=self.args.add_diag)
        else:
            self.up_gate_trans, self.down_trans = None, None

    def _trans_forward(self, x):
        if self.up_gate_trans is not None:
            x_ts = self.up_gate_trans(x)
        else:
            x_ts = x
        up_states = self.up_proj(x_ts, qa_trans=self.up_gate_trans)
        gate_states = self.gate_proj(x_ts, qa_trans=self.up_gate_trans)

        x_act_fn = self.act_fn(gate_states) * up_states
        if self.down_trans is not None:
            x_ts_2 = self.down_trans(x_act_fn)
        else:
            x_ts_2 = x_act_fn
        down_states = self.down_proj(x_ts_2, qa_trans=self.down_trans)
        return down_states

    def _ori_forward(self, x):
        '''origin implement: down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))'''
        if self.diag_init == "sq_style":
            self.up_smax = torch.maximum(self.up_smax, x.reshape(-1, x.shape[-1]).abs().max(0)[0].clone().detach())
        x = self.act_fn(self.gate_proj._ori_forward(x)) * self.up_proj._ori_forward(x)
        if self.diag_init == "sq_style":
            self.down_smax = torch.maximum(self.down_smax, x.reshape(-1, x.shape[-1]).abs().max(0)[0].clone().detach())
        down_states = self.down_proj._ori_forward(x)
        return down_states

    def forward(self, x):
        if self._ori_mode:
            return self._ori_forward(x)
        return self._trans_forward(x)

    def reparameterize(self, ):
        if self.up_gate_trans is not None:
            self.up_gate_trans.to_eval_mode()
            self.down_trans.to_eval_mode()
        self.gate_proj.reparameterize(qa_trans=self.up_gate_trans)
        self.up_proj.reparameterize(qa_trans=self.up_gate_trans)
        self.down_proj.reparameterize(qa_trans=self.down_trans)
        if self.up_gate_trans is not None:
            self.up_gate_trans.use_diag = False
        # merge trans's diag scale
        if self.down_trans is not None and self.down_trans.add_diag:
            up_weight = self.up_proj.linear.weight
            ori_dtype = up_weight.dtype
            up_weight = up_weight.to(torch.float64).T.mul(self.down_trans.diag_scale.to(torch.float64)).T
            self.up_proj.linear.weight.data = up_weight.to(ori_dtype)
            self.down_trans.use_diag = False

    def init_diag_scale(self, alpha=0.5):
        assert hasattr(self, "up_smax") and hasattr(self, "down_smax")
        upw_smax = torch.cat([self.up_proj.linear.weight, self.gate_proj.linear.weight], dim=0).abs().max(dim=0)[0]
        downw_smax = self.down_proj.linear.weight.abs().max(dim=0)[0]
        if self.up_gate_trans is not None:
            self.up_gate_trans.diag_scale.data = get_init_scale(upw_smax, self.up_smax, alpha)
        if self.down_trans is not None:
            self.down_trans.diag_scale.data = get_init_scale(downw_smax, self.down_smax, alpha)
        del self.up_smax, self.down_smax
        self.diag_init = None

    def rep_matrix_only(self, ):
        if self.up_gate_trans is not None:
            self.up_gate_trans.to_eval_mode()
            self.down_trans.to_eval_mode()


class FlatQuantQwen2Attention(Qwen2Attention):
    @staticmethod
    def _build_rotary_embedding(module):
        config = module.config
        device = module.q_proj.weight.device
        attempts = [
            lambda: Qwen2RotaryEmbedding(config=config, device=device),
            lambda: Qwen2RotaryEmbedding(config=config),
            lambda: Qwen2RotaryEmbedding(config, device=device),
            lambda: Qwen2RotaryEmbedding(config),
        ]
        last_error = None
        for attempt in attempts:
            try:
                return attempt()
            except (TypeError, AttributeError) as exc:
                last_error = exc
        if last_error is not None:
            raise last_error
        raise RuntimeError("Unable to construct Qwen2RotaryEmbedding")

    def _compute_position_embeddings(self, value_states, position_ids):
        if not hasattr(self, "rotary_emb"):
            raise AttributeError("FlatQuantQwen2Attention has no rotary_emb fallback")

        attempts = [
            lambda: self.rotary_emb(value_states, position_ids),
            lambda: self.rotary_emb(value_states, position_ids=position_ids),
            lambda: self.rotary_emb(position_ids=position_ids, x=value_states),
        ]
        last_error = None
        for attempt in attempts:
            try:
                return attempt()
            except TypeError as exc:
                last_error = exc
        if last_error is not None:
            raise last_error
        raise RuntimeError("Unable to compute Qwen2 rotary position embeddings")

    def __init__(self, args, module: Qwen2Attention):
        super().__init__(module.config, module.layer_idx)
        self.args = args
        self._use_position_embeddings_api = "position_embeddings" in inspect.signature(module.forward).parameters
        self.hidden_size = getattr(module, "hidden_size", module.config.hidden_size)
        self.num_heads = getattr(module, "num_heads", getattr(module, "num_attention_heads", module.config.num_attention_heads))
        self.num_key_value_heads = getattr(module, "num_key_value_heads", module.config.num_key_value_heads)
        self.head_dim = getattr(module, "head_dim", getattr(module.config, "head_dim", self.hidden_size // self.num_heads))
        self.num_key_value_groups = getattr(module, "num_key_value_groups", self.num_heads // self.num_key_value_heads)
        self.scaling = getattr(module, "scaling", self.head_dim ** -0.5)
        self.attention_dropout = getattr(module, "attention_dropout", getattr(module.config, "attention_dropout", 0.0))
        self.is_causal = getattr(module, "is_causal", True)
        self.sliding_window = getattr(module, "sliding_window", None)
        if hasattr(module, "rotary_emb"):
            self.rotary_emb = module.rotary_emb
        else:
            self.rotary_emb = self._build_rotary_embedding(module)
        
        self.q_proj = FlatQuantizedLinear(args, module.q_proj)
        self.k_proj = FlatQuantizedLinear(args, module.k_proj)
        self.v_proj = FlatQuantizedLinear(args, module.v_proj)
        self.o_proj = FlatQuantizedLinear(args, module.o_proj)
        self.add_fq_trans()

        if args.q_bits < 16:
            self.q_cache_quantizer = ActivationQuantizer(bits=args.q_bits, \
                                        sym=not(args.q_asym), lac=args.lac, groupsize=-1, )
        if args.k_bits < 16:
            self.k_cache_quantizer = ActivationQuantizer(bits=args.k_bits, \
                                        sym=not(args.k_asym), lac=args.lac, groupsize=-1, )
        if args.v_bits < 16:
            self.v_cache_quantizer = ActivationQuantizer(bits=args.v_bits, \
                                        sym=not(args.v_asym), lac=args.lac, groupsize=-1, )

        self._ori_mode = False
        self._eval_mode = False
        self.diag_init = args.diag_init
        if self.diag_init == "sq_style":
            self.ln_smax = torch.ones_like(self.q_proj.linear.weight.abs().max(dim=0)[0]).cuda() * 1e-5

    def add_fq_trans(self):
        if self.args.direct_inv:
            SingleTransMatrix, DecomposeTransMatrix = InvSingleTransMatrix, InvDecomposeTransMatrix
        else:
            SingleTransMatrix, DecomposeTransMatrix = SVDSingleTransMatrix, SVDDecomposeTransMatrix
        if self.args.w_bits < 16 or self.args.a_bits < 16:
            ln_dim_left, ln_dim_right = get_decompose_dim(self.q_proj.linear.weight.shape[1])
            self.ln_trans = DecomposeTransMatrix(ln_dim_left, ln_dim_right, add_diag=self.args.add_diag)
            self.o_trans = SingleTransMatrix(self.config.num_attention_heads)
        else:
            self.ln_trans, self.o_trans = None, None

        head_dim = self.config.hidden_size // self.config.num_attention_heads
        if self.args.k_bits < 16 or self.args.q_bits < 16:
            self.kcache_trans = SingleTransMatrix(head_dim)
        else:
            self.kcache_trans = None
        if self.args.v_bits < 16 or self.args.w_bits < 16 or self.args.a_bits < 16:
            self.vcache_trans = SingleTransMatrix(head_dim)
        else:
            self.vcache_trans = None

    def _trans_forward_after_ln(self, hidden_states):
        if self.ln_trans is not None:
            hidden_states = self.ln_trans(hidden_states)
        query_states = self.q_proj(hidden_states, qa_trans=self.ln_trans)
        key_states = self.k_proj(hidden_states, qa_trans=self.ln_trans)
        if self.args.separate_vtrans:
            value_states = self.v_proj(hidden_states, qa_trans=self.ln_trans)
        else:
            value_states = self.v_proj(hidden_states, qa_trans=self.ln_trans, out_trans=self.vcache_trans)
        return query_states, key_states, value_states

    def _ori_forward_after_ln(self, hidden_states):
        if self.diag_init == "sq_style" and hasattr(self, "ln_smax"):
            self.ln_smax = torch.maximum(self.ln_smax, \
                hidden_states.reshape(-1, hidden_states.shape[-1]).abs().max(0)[0].clone().detach())
        query_states = self.q_proj._ori_forward(hidden_states)
        key_states = self.k_proj._ori_forward(hidden_states)
        value_states = self.v_proj._ori_forward(hidden_states)
        return query_states, key_states, value_states

    def quant_vcache(self, value_states):
        if self.args.separate_vtrans:
            value_states = self.vcache_trans(value_states)
        if self.args.v_bits < 16:
            value_states = self.v_cache_quantizer(value_states)
        return value_states

    def quant_kcache(self, q, k):
        if not (self.args.k_bits < 16 or self.args.q_bits < 16):
            return q, k
        # Q/K transform
        if self.kcache_trans is not None:
            q = self.kcache_trans(q, inv_t=True)
            k = self.kcache_trans(k)
        if self.args.q_bits < 16:
            q = self.q_cache_quantizer(q).to(q)
        # TODO: by default do the per-head quantizaion for k-v-cache
        if self.args.k_bits < 16:
            k = self.k_cache_quantizer(k).to(q)
        return q, k

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,
        past_key_values=None,
        **kwargs,
    ):
        cache_obj = past_key_values if past_key_values is not None else past_key_value
        bsz, q_len, _ = hidden_states.size()
        if self._ori_mode:
            query_states, key_states, value_states = self._ori_forward_after_ln(hidden_states)
        else:
            query_states, key_states, value_states = self._trans_forward_after_ln(hidden_states)

        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        if position_embeddings is None:
            # logger.warning_once(
            #     "The attention layers in this model are transitioning from computing the RoPE embeddings internally "
            #     "through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed "
            #     "`position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be "
            #     "removed and `position_embeddings` will be mandatory."
            # )
            cos, sin = self._compute_position_embeddings(value_states, position_ids)
        else:
            cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
        # ---- here do the quantization ----
        if not self._ori_mode:
            query_states, key_states = self.quant_kcache(query_states, key_states)
            value_states = self.quant_vcache(value_states)

        if cache_obj is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}  # Specific to RoPE models
            key_states, value_states = cache_obj.update(key_states, value_states, self.layer_idx, cache_kwargs)

        # repeat k/v heads if n_kv_heads < n_heads
        key_states = repeat_kv(key_states, self.num_key_value_groups)
        value_states = repeat_kv(value_states, self.num_key_value_groups) # bnsh
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(self.head_dim)

        if attention_mask is not None:  # no matter the length, we just slice it
            causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]
            attn_weights = attn_weights + causal_mask

        # upcast attention to fp32
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_weights = nn.functional.dropout(attn_weights, p=self.attention_dropout, training=self.training)
        attn_output = torch.matmul(attn_weights, value_states)

        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):
            raise ValueError(
                f"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is"
                f" {attn_output.size()}"
            )

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(bsz, q_len, self.hidden_size)
        if self._ori_mode:
            attn_output = self.o_proj._ori_forward(attn_output)
        else:
            # new foward: 
            if self.o_trans is None and self.vcache_trans is not None:
                # attn_output = self.vcache_trans(value_states)
                init_shape = attn_output.shape
                attn_output = attn_output.reshape(-1, self.config.num_attention_heads, self.config.hidden_size//self.config.num_attention_heads)
                attn_output = torch.matmul(attn_output, self.vcache_trans.get_matrix(inv_t=True).T.to(attn_output)).reshape(init_shape)
                attn_output = self.o_proj(attn_output)
            else:
                init_shape = attn_output.shape
                attn_output = attn_output.reshape(-1, self.config.num_attention_heads, self.config.hidden_size//self.config.num_attention_heads)
                attn_output = torch.matmul(self.o_trans.get_matrix().T.to(attn_output), attn_output).reshape(init_shape)
                if not self._eval_mode:
                    attn_o_og_it = self.o_trans.get_matrix(inv_t=True)
                    attn_v_og_it = self.vcache_trans.get_matrix(inv_t=True)
                    attn_output = self.o_proj(attn_output, qa_trans=[attn_o_og_it, attn_v_og_it])
                else:
                    attn_output = self.o_proj(attn_output)

        if not output_attentions:
            attn_weights = None
        return attn_output, attn_weights, cache_obj

    def reparameterize(self):
        if self.ln_trans is not None:
            self.ln_trans.to_eval_mode()
        if self.kcache_trans is not None:
            self.kcache_trans.to_eval_mode()
        if self.vcache_trans is not None:
            self.vcache_trans.to_eval_mode()
        if self.o_trans is not None:
            self.o_trans.to_eval_mode()
        self.q_proj.reparameterize(qa_trans=self.ln_trans)
        self.k_proj.reparameterize(qa_trans=self.ln_trans)
        if self.args.separate_vtrans:
            self.v_proj.reparameterize(qa_trans=self.ln_trans)
        else:
            self.v_proj.reparameterize(qa_trans=self.ln_trans, out_trans=self.vcache_trans)
        if self.o_trans is not None and self.vcache_trans is not None:
            attn_o_og_it = self.o_trans.get_matrix(inv_t=True)
            attn_v_og_it = self.vcache_trans.get_matrix(inv_t=True)
            self.o_proj.reparameterize(qa_trans=[attn_o_og_it, attn_v_og_it])
        self._eval_mode = True

    def init_diag_scale(self, alpha=0.5):
        assert hasattr(self, "ln_smax")
        qkvw_smax = torch.cat([self.q_proj.linear.weight, self.k_proj.linear.weight, self.v_proj.linear.weight], dim=0).abs().max(dim=0)[0]
        if self.ln_trans is not None:
            self.ln_trans.diag_scale.data = get_init_scale(qkvw_smax, self.ln_smax, alpha)
        del self.ln_smax
        self.diag_init = None

    def rep_matrix_only(self, ):
        if self.ln_trans is not None:
            self.ln_trans.to_eval_mode()
        if self.kcache_trans is not None:
            self.kcache_trans.to_eval_mode()
        if self.vcache_trans is not None:
            self.vcache_trans.to_eval_mode()
        if self.o_trans is not None:
            self.o_trans.to_eval_mode()


def apply_flatquant_to_qwen(args, model):
    skip_initialization()
    # Replace module with FlatQuant version
    for layer in range(model.config.num_hidden_layers):
        # attn
        model.model.layers[layer].self_attn = FlatQuantQwen2Attention(args, model.model.layers[layer].self_attn)
        # mlp
        model.model.layers[layer].mlp = FlatQuantQwen2MLP(args, model.model.layers[layer].mlp)
    return model


In [ ]:
%%bash
mkdir -p smart-flip/flatquant
rm -f smart-flip/flatquant/data_utils.py


In [ ]:
%%writefile smart-flip/flatquant/data_utils.py
import os
import pickle
import datasets
import random
import transformers


C4_TRAIN_URL = "https://huggingface.co/datasets/allenai/c4/resolve/main/en/c4-train.00000-of-01024.json.gz"
C4_VALIDATION_URL = "https://huggingface.co/datasets/allenai/c4/resolve/main/en/c4-validation.00000-of-00008.json.gz"


class TokenizerWrapper:
    def __init__(self, input_ids):
        self.input_ids = input_ids


def get_wikitext2(nsamples, seqlen, tokenizer, eval_mode=False):
    if eval_mode:
        testdata = datasets.load_dataset('./datasets/wikitext', 'wikitext-2-raw-v1', split='test')
        testenc = tokenizer("\n\n".join(testdata['text']), return_tensors='pt')
        return testenc
    else:
        traindata = datasets.load_dataset('./datasets/wikitext', 'wikitext-2-raw-v1', split='train')
        traindata = traindata.filter(lambda x: len(x) > 0)
        traindata = traindata.map(lambda x : {'text': x['text'].strip()})
        trainenc = tokenizer("\n\n".join(traindata['text']), return_tensors='pt')    
        trainloader = []
        for _ in range(nsamples):
            i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
            j = i + seqlen
            inp = trainenc.input_ids[:, i:j]
            tar = inp.clone()
            tar[:, :-1] = -100
            trainloader.append((inp, tar))
        return trainloader


def get_c4_new(nsamples, seqlen, tokenizer, eval_mode=False):
    split = "validation" if eval_mode else "train"
    url = C4_VALIDATION_URL if eval_mode else C4_TRAIN_URL
    dataset = datasets.load_dataset(
        "json",
        data_files={split: url},
        split=split,
        streaming=True,
    )

    if eval_mode:
        texts = []
        for item in dataset:
            text = item.get("text", "").strip()
            if not text:
                continue
            texts.append(text)
            if len(texts) >= 1100:
                break
        valenc = tokenizer(' '.join(texts), return_tensors='pt')
        valenc = valenc.input_ids[:, :(256 * seqlen)]
        valenc = TokenizerWrapper(valenc)
        return valenc

    trainloader = []
    for item in dataset:
        text = item.get("text", "").strip()
        if not text:
            continue
        trainenc = tokenizer(text, return_tensors='pt')
        if trainenc.input_ids.shape[1] < seqlen:
            continue
        max_start = trainenc.input_ids.shape[1] - seqlen
        start = random.randint(0, max_start)
        end = start + seqlen
        inp = trainenc.input_ids[:, start:end]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
        if len(trainloader) >= nsamples:
            break

    if len(trainloader) < nsamples:
        raise RuntimeError(f"Unable to collect {nsamples} C4 samples; only found {len(trainloader)} usable documents")
    return trainloader


def get_ptb_new(nsamples, seqlen, tokenizer, eval_mode=False):
    if eval_mode:
        testdata = datasets.load_dataset('./datasets/ptb_text_only', 'penn_treebank', split='test')
        testenc = tokenizer(" ".join(testdata['sentence']), return_tensors='pt')
        return testenc
    else:
        traindata = datasets.load_dataset('./datasets/ptb_text_only', 'penn_treebank', split='train')
        trainenc = tokenizer(" ".join(traindata['sentence']), return_tensors='pt')
        trainloader = []
        for _ in range(nsamples):
            i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
            j = i + seqlen
            inp = trainenc.input_ids[:, i:j]
            tar = inp.clone()
            tar[:, :-1] = -100
            trainloader.append((inp, tar))
        return trainloader


def get_pile(nsamples, seqlen, tokenizer):
    traindata = datasets.load_dataset("./datasets/pile-val-backup", split="validation")
    trainenc = tokenizer("\n\n".join(traindata['text'][:1000]), return_tensors='pt')
    trainloader = []
    for _ in range(nsamples):
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
    return trainloader


def get_loaders(
    args, name, tokenizer, nsamples=128, seqlen=2048, eval_mode=False
):
    if 'wikitext2' in name:
        dataset = get_wikitext2(nsamples, seqlen, tokenizer, eval_mode)
    elif 'ptb' in name:
        dataset = get_ptb_new(nsamples, seqlen, tokenizer, eval_mode)
    elif 'c4' in name:
        dataset = get_c4_new(nsamples, seqlen, tokenizer, eval_mode)
    elif 'pile' in name:
        dataset = get_pile(nsamples, seqlen, tokenizer)

    if 'c4' in name and eval_mode:
        dataset = dataset.input_ids
        dataset = TokenizerWrapper(dataset)
    return dataset


In [ ]:
%%bash
mkdir -p smart-flip/flatquant
rm -f smart-flip/flatquant/flatness.py


In [ ]:
%%writefile smart-flip/flatquant/flatness.py
import os
import math
import functools
from tqdm import tqdm
import transformers

import torch
import numpy as np
from numpy import linalg as LA
from matplotlib import pyplot as plt
from brokenaxes import brokenaxes

import flatquant.utils as utils
import flatquant.model_utils as model_utils
import flatquant.data_utils as data_utils
import flatquant.flat_utils as flat_utils
import flatquant.hadamard_utils as hadamard_utils
import flatquant.quant_utils as quant_utils


@torch.no_grad()
def get_act_stats(model, dataset):
    model.eval()
    device = next(model.parameters()).device

    # activations
    inps = {}
    # weights
    weights = {}

    target_layer_type = torch.nn.Linear

    def stat_tensor(name, m, x, y):
        if not name in inps:
            inps[name] = []
            weights[name] = []
        w = m.weight
        inps[name].append(x.float().cpu())
        weights[name] = w.float().cpu()

    def stat_input_hook(m, x, y, name):
        if isinstance(x, tuple):
            x = x[0]
        if isinstance(y, tuple):
            y = y[0]
        stat_tensor(name, m, x, y)

    hooks = []
    for name, m in model.named_modules():
        if isinstance(m, target_layer_type) and not "lm_head" in name:
            hooks.append(
                m.register_forward_hook(
                    functools.partial(stat_input_hook, name=name))
            )

    for data in dataset:
        model(data[0].to(device))
        break

    for name in list(inps.keys()):
        in_dim = inps[name][0].shape[-1]
        inps[name] = inps[name][0].reshape(-1, in_dim)

    for h in hooks:
        h.remove()

    return inps, weights


@torch.no_grad()
def get_flatness(args, logger, transform_type=None):
    model, apply_flatquant_to_model = model_utils.get_model(args.model, args.hf_token)
    tokenizer = transformers.AutoTokenizer.from_pretrained(args.model, use_fast=False, token=args.hf_token)
    model.eval()

    # get calibration data
    trainloader = data_utils.get_loaders(
        args, args.cali_dataset, tokenizer, 
        nsamples=args.nsamples, eval_mode=False
    )
    logger.info("Finished loading training data.")

    # apply pre-quantization transformations
    if transform_type is not None:
        if transform_type == "flatquant":
            args.w_bits = 4; args.a_bits = 4
            model = apply_flatquant_to_model(args, model)
            logger.info("Finished applying FlatQuant to model.")
            flat_utils.load_flat_matrices(args, model, path=args.matrix_path)
            flat_utils.reparameterize_model(model)
            logger.info("Finished reparameterize model.")
            quant_utils.set_quantizer_state(model, enable=False)
        elif transform_type == "hadamard":
            pass
        elif transform_type == "smoothquant":
            args.act_scales_path = os.path.join(f"./act_scales/{args.model_name}.pt")
            args.act_scales = torch.load(args.act_scales_path)
            args.smooth_alpha = 0.85
        else:
            raise NotImplementedError        

    if args.distribute_model:
        utils.distribute_model(model)
    else:
        model.to(utils.DEV)

    inps, weights = get_act_stats(model, trainloader)
    flatness = {}
    for name in tqdm(inps.keys()):
        x = inps[name]
        w = weights[name]

        if transform_type is None or transform_type == "flatquant":
            x_flatness = LA.norm(x.cpu().numpy(), axis=0)
            w_flatness = LA.norm(w.cpu().numpy(), axis=0)
            flatness[name] = {
                "x": x_flatness, "w": w_flatness
            }
        elif transform_type == "hadamard":
            hidden_dim = x.shape[-1]
            if hadamard_utils.is_pow2(hidden_dim):
                had = hadamard_utils.get_had_pow2(hidden_dim).cuda()
                x_had = x.cuda() @ had
                w_had = w.cuda() @ had
            else:
                had_right, k = hadamard_utils.get_hadK(hidden_dim)
                had_right = had_right.cuda() / math.sqrt(k)
                had_left = hadamard_utils.get_had_pow2(hidden_dim // k).cuda()
                x_had = flat_utils.kronecker_matmul(x.cuda(), had_left, had_right)
                w_had = flat_utils.kronecker_matmul(w.cuda(), had_left, had_right)
            x_had_flatness = LA.norm(x_had.cpu().numpy(), axis=0)
            w_had_flatness = LA.norm(w_had.cpu().numpy(), axis=0)
            flatness[name] = {
                "x": x_had_flatness, "w": w_had_flatness
            }
        elif transform_type == "smoothquant":
            act_scales = args.act_scales[name].to(x.device)
            weight_scales = w.abs().max(dim=0)[0].clamp(min=1e-5)
            scales = (
                (act_scales.pow(args.smooth_alpha) / weight_scales.pow(1 - args.smooth_alpha))
                .clamp(min=1e-5)
            )
            x_sq = x / scales
            w_sq = w * scales
            x_sq_flatness = LA.norm(x_sq.cpu().numpy(), axis=0)
            w_sq_flatness = LA.norm(w_sq.cpu().numpy(), axis=0)
            flatness[name] = {
                "x": x_sq_flatness, "w": w_sq_flatness
            }

    args.cache_dir = os.path.join(args.vis_dir, ".cache")
    os.makedirs(args.cache_dir, exist_ok=True)
    torch.save(flatness, os.path.join(args.cache_dir, f"flatness_{transform_type}.pt" if transform_type is not None else f"flatness.pt"))
    logger.info(f"Flatness stats saved at {args.cache_dir}.")

    del model
    utils.cleanup_memory()

    return flatness


def get_act_scales(args, logger):
    model, apply_flatquant_to_model = model_utils.get_model(args.model, args.hf_token)
    tokenizer = transformers.AutoTokenizer.from_pretrained(args.model, use_fast=False, token=args.hf_token)
    model.eval()

    # get calibration data
    dataset = data_utils.get_loaders(
        args, args.cali_dataset, tokenizer, 
        nsamples=args.nsamples, eval_mode=False
    )
    logger.info("Finished loading training data.")

    if args.distribute_model:
        utils.distribute_model(model)
    else:
        model.to(utils.DEV)
    device = next(model.parameters()).device

    act_scales = {}

    target_layer_type = torch.nn.Linear

    def stat_tensor(name, tensor):
        hidden_dim = tensor.shape[-1]
        tensor = tensor.view(-1, hidden_dim).abs().detach()
        comming_max = torch.max(tensor, dim=0)[0].float().cpu()
        if name in act_scales:
            act_scales[name] = torch.max(act_scales[name], comming_max)
        else:
            act_scales[name] = comming_max

    def stat_input_hook(m, x, y, name):
        if isinstance(x, tuple):
            x = x[0]
        stat_tensor(name, x)

    hooks = []
    for name, m in model.named_modules():
        if isinstance(m, target_layer_type) and not "lm_head" in name:
            hooks.append(
                m.register_forward_hook(
                    functools.partial(stat_input_hook, name=name))
            )

    for i in tqdm(range(len(dataset))):
        input_ids = dataset[i][0].to(device)
        model(input_ids)

    for h in hooks:
        h.remove()

    args.act_scales_dir = "./act_scales"
    os.makedirs("./act_scales", exist_ok=True)
    torch.save(act_scales, os.path.join(args.act_scales_dir, f"{args.model_name}.pt"))
    logger.info(f"Activation scales saved at {args.act_scales_dir}.")

    del model
    utils.cleanup_memory()

    return act_scales


def plot_flatness(args, name, vectors, vector_names, y_max=None, y_broken=None, label_pad=15):
    for i in range(len(vectors)):
        sorted_indices = sorted(range(len(vectors[i])), key=lambda k: abs(vectors[i][k]), reverse=True)
        vectors[i] = [vectors[i][j] for j in sorted_indices]

    colors = ["#f44336", "#0288d1", "#8bc34a", "#808080"]
    fontsize = 20
    label_fontsize = 25
    linewidth = 4
    step_cnt = 100
    if y_broken is None:
        fig, ax1 = plt.subplots(1, 1)
    else:
        ax1 = brokenaxes(
            ylims=((0, y_broken[0]), (y_broken[1], y_max)),
            hspace=0.03,
        )

    # plot weight distribution
    for i in range(len(vectors)):
        x = np.linspace(0, len(vectors[i]) - 1, step_cnt)
        y = np.interp(x, range(len(vectors[i])), vectors[i])
        ax1.plot(x, y, color=colors[i], linewidth=linewidth, zorder=1000*(len(vectors) - i), label=vector_names[i])
    
    ax1.set_ylabel("Magnitude", fontsize=label_fontsize, labelpad=fontsize+label_pad if y_broken is not None else None)
    ax1.set_xlabel("Channels", fontsize=label_fontsize, labelpad=fontsize+8 if y_broken is not None else None)
    ax1.grid(axis='x', linestyle='--')
    ax1.grid(axis='y', linestyle='--')
    ax1.tick_params(axis="x", labelsize=fontsize-2)
    ax1.tick_params(axis="y", labelsize=fontsize-2)
    ax1.legend(loc="upper right", fontsize=fontsize-2)
    if y_broken is not None:
        for spine in ax1.spines["top"]:
            spine.set_visible(True)
        for spine in ax1.spines["right"]:
            spine.set_visible(True)
    elif y_max is not None:
        ax1.set_ylim(0, y_max)
    else:
        ax1.set_ylim(bottom=0)

    plt.tight_layout()
    if y_broken is not None:
        for handle in ax1.diag_handles:
            handle.remove()
        ax1.draw_diags()

    plt.savefig(os.path.join(args.vis_dir, f"{name}.pdf"))
    plt.close()


def plot_flatness_all_layers(args, flatness_flatquant, flatness_hadamard,
                             flatness_smoothquant, flatness_vanilla):
    for name in flatness_vanilla.keys():
        xw_flatnesses = [flatness_flatquant[name + ".linear"], flatness_hadamard[name], flatness_smoothquant[name], flatness_vanilla[name]]
        flatness_names = ["FlatQuant", "Hadamard", "SmoothQuant", "Vanilla"]
        x_flatnesses = [xw_flatness["x"] for xw_flatness in xw_flatnesses]
        w_flatnesses = [xw_flatness["w"] for xw_flatness in xw_flatnesses]
        plot_flatness(args, name + ".x", x_flatnesses, flatness_names)
        plot_flatness(args, name + ".w", w_flatnesses, flatness_names)


In [ ]:
%%bash
mkdir -p smart-flip/flatquant
rm -f smart-flip/flatquant/model_utils.py


In [ ]:
%%writefile smart-flip/flatquant/model_utils.py
import torch
import transformers
import logging
from flatquant.utils import skip
from flatquant.model_tools.llama_utils import apply_flatquant_to_llama
from flatquant.model_tools.llama31_utils import apply_flatquant_to_llama_31


def skip_initialization():
    torch.nn.init.kaiming_uniform_ = skip
    torch.nn.init.uniform_ = skip
    torch.nn.init.normal_ = skip


def _hf_auth_kwargs(hf_token):
    if not hf_token:
        return {}
    return {"token": hf_token}


def get_llama(model_name, hf_token):
    skip_initialization()
    config = transformers.LlamaConfig.from_pretrained(model_name, **_hf_auth_kwargs(hf_token))
    config._attn_implementation_internal = "eager"
    model = transformers.LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype='auto',
        config=config,
        low_cpu_mem_usage=True,
        **_hf_auth_kwargs(hf_token),
    )
    model.seqlen = 2048
    logging.info(f'---> Loading {model_name} Model with seq_len: {model.seqlen}')
    return model, apply_flatquant_to_llama


def get_llama_31(model_name, hf_token):
    skip_initialization()
    config = transformers.LlamaConfig.from_pretrained(model_name, **_hf_auth_kwargs(hf_token))
    config._attn_implementation_internal = "eager"
    model = transformers.LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype='auto',
        config=config,
        low_cpu_mem_usage=True,
        **_hf_auth_kwargs(hf_token),
    )
    model.seqlen = 2048
    logging.info(f'---> Loading {model_name} Model with seq_len: {model.seqlen}')
    return model, apply_flatquant_to_llama_31


def get_qwen2(model_name, hf_token):
    skip_initialization()
    try:
        from transformers import Qwen2ForCausalLM
    except ImportError:
        logging.error("Qwen2 model is not available in this version of 'transformers'. Please update the library.")
        raise ImportError("Qwen2 model is not available. Ensure you're using a compatible version of the 'transformers' library.")

    config = transformers.Qwen2Config.from_pretrained(model_name, **_hf_auth_kwargs(hf_token))
    config._attn_implementation_internal = "eager"
    model = Qwen2ForCausalLM.from_pretrained(
        model_name,
        torch_dtype='auto',
        config=config,
        low_cpu_mem_usage=True,
        **_hf_auth_kwargs(hf_token),
    )
    model.seqlen = 2048
    logging.info(f'---> Loading {model_name} Model with seq_len: {model.seqlen}')

    from flatquant.model_tools.qwen_utils import apply_flatquant_to_qwen
    return model, apply_flatquant_to_qwen


def get_mistral(model_name, hf_token):
    skip_initialization()
    try:
        from transformers import MistralForCausalLM, MistralConfig
    except ImportError:
        logging.error("Mistral model is not available in this version of 'transformers'. Please update the library.")
        raise ImportError("Mistral model is not available. Ensure you're using a compatible version of the 'transformers' library.")

    config = MistralConfig.from_pretrained(model_name, **_hf_auth_kwargs(hf_token))
    config._attn_implementation_internal = "eager"
    model = MistralForCausalLM.from_pretrained(
        model_name,
        torch_dtype='auto',
        config=config,
        low_cpu_mem_usage=True,
        **_hf_auth_kwargs(hf_token),
    )
    model.seqlen = 2048
    logging.info(f'---> Loading {model_name} Model with seq_len: {model.seqlen}')

    from src.quantization.flatquant_mistral import apply_flatquant_to_mistral
    return model, apply_flatquant_to_mistral


def get_opt(model_name):
    skip_initialization()
    model = transformers.OPTForCausalLM.from_pretrained(model_name,
                                                        torch_dtype='auto',
                                                        low_cpu_mem_usage=True)
    model.seqlen = model.config.max_position_embeddings
    logging.info(f'---> Loading {model_name} Model with seq_len: {model.seqlen}')
    raise NotImplementedError("Post-processing for OPT model is not implemented yet.")


# Unified model loading function
def get_model(model_name, hf_token=None):
    lower_name = model_name.lower()
    if 'llama-3.1' in lower_name:
        return get_llama_31(model_name, hf_token)
    elif 'llama' in lower_name:
        return get_llama(model_name, hf_token)
    elif 'qwen2.5' in lower_name or 'qwen-2.5' in lower_name:
        return get_qwen2(model_name, hf_token)
    elif 'mistral' in lower_name:
        return get_mistral(model_name, hf_token)
    else:
        raise ValueError(f'Unknown model {model_name}')


In [ ]:
%%bash
mkdir -p smart-flip/flatquant
rm -f smart-flip/flatquant/trans_utils.py


In [ ]:
%%writefile smart-flip/flatquant/trans_utils.py
import os

import torch
import torch.nn as nn

from flatquant.flat_utils import kronecker_matmul
from flatquant.function_utils import get_init_weight, get_inverse


def _env_flag(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() not in {"0", "false", "no", "off"}


FLATQUANT_ORTHOGONAL_MAP = os.environ.get("FLATQUANT_ORTHOGONAL_MAP", "matrix_exp")
FLATQUANT_USE_TRIVIALIZATION = _env_flag("FLATQUANT_USE_TRIVIALIZATION", True)


def _orthogonal(linear: nn.Linear) -> nn.Module:
    return nn.utils.parametrizations.orthogonal(
        linear,
        orthogonal_map=FLATQUANT_ORTHOGONAL_MAP,
        use_trivialization=FLATQUANT_USE_TRIVIALIZATION,
    )


# ---------- transformation version of singular value decomposition ----------
class SVDSingleTransMatrix(nn.Module):
    def __init__(self, size):
        super(SVDSingleTransMatrix, self).__init__()
        self.linear_u = nn.Linear(size, size, bias=False, dtype=torch.float32)
        self.linear_u.weight.data = get_init_weight(size).to(self.linear_u.weight)
        self.linear_u = _orthogonal(self.linear_u)
        self.linear_v = nn.Linear(size, size, bias=False, dtype=torch.float32)
        self.linear_v.weight.data = get_init_weight(size).to(self.linear_v.weight)
        self.linear_v = _orthogonal(self.linear_v)
        self.linear_diag = torch.nn.Parameter(torch.ones(size, dtype=torch.float32), requires_grad=True)

        self._eval_mode = False

    def forward(self, inp, inv_t=False):
        init_shape = inp.shape
        matirx = self.get_matrix(inv_t=inv_t).to(inp)
        inp = inp.reshape(-1, matirx.shape[0])
        return inp.matmul(matirx).reshape(init_shape)

    def get_matrix(self, inv_t=False):
        if not self._eval_mode:
            orthog_u, orthog_v = self.linear_u.weight, self.linear_v.weight
            linear_diag = self.linear_diag
            if inv_t:
                linear_diag = 1 / linear_diag
            return orthog_u @ torch.diag(linear_diag) @ orthog_v.t()
        else:
            if inv_t:
                return self.matrix_inv_t
            return self.matrix

    def to_eval_mode(self):
        if not self._eval_mode:
            matrix = self.linear_u.weight @ torch.diag(self.linear_diag) @ self.linear_v.weight.t()
            matrix_inv_t = self.linear_u.weight @ torch.diag(1 / self.linear_diag) @ self.linear_v.weight.t()
            self.matrix = nn.Parameter(matrix, requires_grad=False)
            self.matrix_inv_t = nn.Parameter(matrix_inv_t, requires_grad=False)
            self._eval_mode = True
            del self.linear_u, self.linear_diag, self.linear_v

    def __repr__(self):
        res = f"SVDSingleTransMatrix(eval_mode={self._eval_mode}"
        if hasattr(self, 'matrix'):
            res += f", matrix.shape={self.matrix.shape})"
        else:
            res += f", matrix.shape={self.linear_u.weight.shape})"
        return res


class SVDDecomposeTransMatrix(nn.Module):
    def __init__(self, left_size, right_size, add_diag=False, diag_init_para=None):
        super(SVDDecomposeTransMatrix, self).__init__()
        self.linear_u_left = nn.Linear(left_size, left_size, bias=False, dtype=torch.float32)
        self.linear_u_left.weight.data = get_init_weight(left_size).to(self.linear_u_left.weight)
        self.linear_u_left = _orthogonal(self.linear_u_left)
        self.linear_v_left = nn.Linear(left_size, left_size, bias=False, dtype=torch.float32)
        self.linear_v_left.weight.data = get_init_weight(left_size).to(self.linear_v_left.weight)
        self.linear_v_left = _orthogonal(self.linear_v_left)
        self.linear_diag_left = torch.nn.Parameter(torch.ones(left_size, dtype=torch.float32), requires_grad=True)

        self.linear_u_right = nn.Linear(right_size, right_size, bias=False, dtype=torch.float32)
        self.linear_u_right.weight.data = get_init_weight(right_size).to(self.linear_u_right.weight)
        self.linear_u_right = _orthogonal(self.linear_u_right)
        self.linear_v_right = nn.Linear(right_size, right_size, bias=False, dtype=torch.float32)
        self.linear_v_right.weight.data = get_init_weight(right_size).to(self.linear_v_right.weight)
        self.linear_v_right = _orthogonal(self.linear_v_right)
        self.linear_diag_right = torch.nn.Parameter(torch.ones(right_size, dtype=torch.float32), requires_grad=True)

        self.add_diag = add_diag
        self.use_diag = True
        if self.add_diag:
            if diag_init_para is None:
                self.diag_scale = torch.nn.Parameter(torch.ones((left_size * right_size), dtype=torch.float32), requires_grad=True)
            else:
                self.diag_scale = torch.nn.Parameter(diag_init_para, requires_grad=True)
        self._eval_mode = False

    def forward(self, inp, inv_t=False):
        if self.add_diag and self.use_diag:
            if inv_t:
                inp = inp / self.diag_scale.to(inp)
            else:
                inp = inp * self.diag_scale.to(inp)
        if not self._eval_mode:
            matrix_u_left, matrix_u_right = self.linear_u_left.weight, self.linear_u_right.weight
            matrix_v_left, matrix_v_right = self.linear_v_left.weight, self.linear_v_right.weight
            linear_diag_left, linear_diag_right = self.linear_diag_left,  self.linear_diag_right
            if inv_t:
                linear_diag_left, linear_diag_right = 1 / linear_diag_left, 1 / linear_diag_right
        else:
            matrix_left, matrix_right = self.matrix_left, self.matrix_right
            if inv_t:
                matrix_left, matrix_right = self.matrix_left_inv, self.matrix_right_inv
            return kronecker_matmul(inp, matrix_left.to(inp), matrix_right.to(inp))
        matrix_left, matrix_right = matrix_u_left @ torch.diag(linear_diag_left) @ matrix_v_left.t(), matrix_u_right @ torch.diag(linear_diag_right) @ matrix_v_right.t()
        return kronecker_matmul(inp, matrix_left.to(inp), matrix_right.to(inp))

    def to_eval_mode(self):
        if not self._eval_mode:
            matrix_left = self.linear_u_left.weight @ torch.diag(self.linear_diag_left) @ self.linear_v_left.weight.t()
            matrix_right = self.linear_u_right.weight @ torch.diag(self.linear_diag_right) @ self.linear_v_right.weight.t()
            matrix_left_inv = self.linear_u_left.weight @ torch.diag(1 / self.linear_diag_left) @ self.linear_v_left.weight.t()
            matrix_right_inv = self.linear_u_right.weight @ torch.diag(1 / self.linear_diag_right) @ self.linear_v_right.weight.t()
            self.matrix_left = nn.Parameter(matrix_left, requires_grad=False)
            self.matrix_right = nn.Parameter(matrix_right, requires_grad=False)
            self.matrix_left_inv = nn.Parameter(matrix_left_inv, requires_grad=False)
            self.matrix_right_inv = nn.Parameter(matrix_right_inv, requires_grad=False)
            del self.linear_u_left, self.linear_diag_left, self.linear_v_left, self.linear_u_right, self.linear_diag_right, self.linear_v_right
            self._eval_mode = True

    def __repr__(self):
        res = f"SVDDecomposeTransMatrix(_eval_mode={self._eval_mode}"
        if hasattr(self, 'matrix_left'):
            res += f", matrix.shape={self.matrix_left.shape}, matrix_right.shape={self.matrix_right.shape}, )"
        else:
            res += f", matrix.shape={self.linear_u_left.weight.shape}, linear_right.shape={self.linear_u_right.weight.shape}, )"
        return res


# ---------- transformation version of direct inverse ----------
class InvSingleTransMatrix(nn.Module):
    def __init__(self, size):
        super(InvSingleTransMatrix, self).__init__()
        linear = nn.Linear(size, size, bias=False)
        linear.weight.data = get_init_weight(size).to(linear.weight)
        self.linear = linear
        self._eval_mode = False

    def forward(self, inp, inv_t=False):
        init_shape = inp.shape
        matirx = self.get_matrix(inv_t=inv_t).to(inp)
        inp = inp.reshape(-1, matirx.shape[0])
        return inp.matmul(matirx).reshape(init_shape)

    def get_matrix(self, inv_t=False):
        if not self._eval_mode:
            matrix = self.linear.weight
            if inv_t:
                matrix = get_inverse(matrix).T
            return matrix
        else:
            if inv_t:
                return self.matrix_inv_t
            return self.matrix

    def to_eval_mode(self):
        if not self._eval_mode:
            matrix = self.linear.weight
            matrix_inv_t = get_inverse(matrix).T
            self.matrix = nn.Parameter(matrix, requires_grad=False)
            self.matrix_inv_t = nn.Parameter(matrix_inv_t, requires_grad=False)
            self._eval_mode = True

    def __repr__(self):
        res = f"InvSingleTransMatrix(eval_mode={self._eval_mode}"
        if hasattr(self, 'matrix'):
            res += f", matrix.shape={self.matrix.shape})"
        else:
            res += f", matrix.shape={self.linear.weight.shape})"
        return res


class InvDecomposeTransMatrix(nn.Module):
    def __init__(self, left_size, right_size, add_diag=False, diag_init_para=None):
        super(InvDecomposeTransMatrix, self).__init__()
        linear_left = nn.Linear(left_size, left_size, bias=False)
        linear_left.weight.data = get_init_weight(left_size).to(linear_left.weight)
        self.linear_left = linear_left

        linear_right = nn.Linear(right_size, right_size, bias=False)
        linear_right.weight.data = get_init_weight(right_size).to(linear_right.weight)
        self.linear_right = linear_right

        self.add_diag = add_diag
        self.use_diag = True
        if self.add_diag:
            if diag_init_para is None:
                self.diag_scale = torch.nn.Parameter(torch.ones((left_size * right_size)), requires_grad=True)
            else:
                self.diag_scale = torch.nn.Parameter(diag_init_para, requires_grad=True)
        self._eval_mode = False

    def forward(self, inp, inv_t=False):
        if self.add_diag and self.use_diag:
            if inv_t:
                inp = inp / self.diag_scale.to(inp)
            else:
                inp = inp * self.diag_scale.to(inp)
        if not self._eval_mode:
            matrix_left, matrix_right = self.linear_left.weight, self.linear_right.weight
            if inv_t:
                matrix_left, matrix_right = get_inverse(matrix_left).T, get_inverse(matrix_right).T
        else:
            matrix_left, matrix_right = self.matrix_left, self.matrix_right
            if inv_t:
                matrix_left, matrix_right = self.matrix_left_inv, self.matrix_right_inv
        return kronecker_matmul(inp, matrix_left.to(inp), matrix_right.to(inp))

    def to_eval_mode(self):
        if not self._eval_mode:
            self.matrix_left = nn.Parameter(self.linear_left.weight, requires_grad=False)
            self.matrix_right = nn.Parameter(self.linear_right.weight, requires_grad=False)
            self.matrix_left_inv = nn.Parameter(get_inverse(self.linear_left.weight).T, requires_grad=False)
            self.matrix_right_inv = nn.Parameter(get_inverse(self.linear_right.weight).T, requires_grad=False)
            del self.linear_left, self.linear_right
            self._eval_mode = True

    def __repr__(self):
        res = f"InvDecomposeTransMatrix(_eval_mode={self._eval_mode}"
        if hasattr(self, 'matrix_left'):
            res += f", matrix.shape={self.matrix_left.shape}, matrix_right.shape={self.matrix_right.shape}, )"
        else:
            res += f", matrix.shape={self.linear_left.weight.shape}, linear_right.shape={self.linear_right.weight.shape}, )"
        return res


In [ ]:
%%bash
mkdir -p smart-flip/scripts/bash
rm -f smart-flip/scripts/bash/egbc_c4_b3_resume_bc_extra_from_llama.sh


In [ ]:
%%writefile smart-flip/scripts/bash/egbc_c4_b3_resume_bc_extra_from_llama.sh
#!/usr/bin/env bash
set -euo pipefail

# Resume the interrupted +BC extra phase after Mistral AWQ/FQ finished.
# Runs only:
#   - Llama3.1 FlatQuant +BC
#   - Llama3 FlatQuant +BC
# Skips a worker if its expected eval JSON already exists.

SCRIPT_DIR="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd)"
RUNNER="$SCRIPT_DIR/egbc_c4_b3.sh"

GPU="${GPU:-0}"
GPU_LIST="${GPU_LIST:-$GPU,$GPU,$GPU,$GPU}"
PYTHON_BIN="${PYTHON_BIN:-python}"
WANDB_PROJECT="${WANDB_PROJECT:-egbc_c4_b3}"
RESULTS_MODELS_DIR="${RESULTS_MODELS_DIR:-./results/models}"
RESULTS_EVAL_DIR="${RESULTS_EVAL_DIR:-./results/eval}"
HF_DATASETS_CACHE="${HF_DATASETS_CACHE:-./data/cache/hf_datasets/datasets-egbc-c4-b3}"
LOG_DIR="${LOG_DIR:-./logs/egbc_c4_b3_resume}"
MAX_ATTEMPTS="${MAX_ATTEMPTS:-3}"
RETRY_SLEEP_SECONDS="${RETRY_SLEEP_SECONDS:-120}"
RUN_FQ_RAW_ONCE="${RUN_FQ_RAW_ONCE:-1}"

mkdir -p "$LOG_DIR" "$HF_DATASETS_CACHE"

flatquant_raw_dir() {
  local model_path="$1"
  local model_slug="${model_path##*/}"
  printf '%s/flatquant_raw/flatquant_raw_%s' "$RESULTS_MODELS_DIR" "$model_slug"
}

run_fq_worker_with_retry() {
  local worker="$1"
  local model_path="$2"
  local expected_json="$3"
  local raw_flag=0
  local raw_dir
  raw_dir="$(flatquant_raw_dir "$model_path")"

  if [[ -f "$expected_json" ]]; then
    echo "Skipping $worker; found $expected_json"
    return 0
  fi

  if [[ ! -f "$raw_dir/flat_parameters.pth" && "$RUN_FQ_RAW_ONCE" == "1" ]]; then
    raw_flag=1
  fi

  local attempt=1
  while (( attempt <= MAX_ATTEMPTS )); do
    echo "================================================================"
    echo "Resume worker: $worker attempt=$attempt/$MAX_ATTEMPTS"
    echo "GPU=$GPU GPU_LIST=$GPU_LIST WANDB_PROJECT=$WANDB_PROJECT RUN_FQ_RAW=$raw_flag"
    echo "expected_json=$expected_json"
    echo "raw_dir=$raw_dir"
    echo "================================================================"

    set +e
    env \
      GPU_LIST="$GPU_LIST" \
      PYTHON_BIN="$PYTHON_BIN" \
      WANDB_PROJECT="$WANDB_PROJECT" \
      RESULTS_MODELS_DIR="$RESULTS_MODELS_DIR" \
      RESULTS_EVAL_DIR="$RESULTS_EVAL_DIR" \
      HF_DATASETS_CACHE="$HF_DATASETS_CACHE" \
      HF_DATASETS_OFFLINE="${HF_DATASETS_OFFLINE:-0}" \
      WORKER="$worker" \
      RUN_FQ_RAW="$raw_flag" \
      bash "$RUNNER" 2>&1 | tee "$LOG_DIR/${worker}.attempt${attempt}.log"
    status="${PIPESTATUS[0]}"
    set -e

    if [[ "$status" == "0" ]]; then
      return 0
    fi

    if [[ -f "$expected_json" ]]; then
      echo "$worker produced $expected_json despite non-zero exit; treating as complete."
      return 0
    fi

    if (( attempt == MAX_ATTEMPTS )); then
      echo "FAILED: $worker after $MAX_ATTEMPTS attempts" >&2
      return "$status"
    fi

    echo "Worker $worker failed with status $status; sleeping ${RETRY_SLEEP_SECONDS}s before retry..."
    sleep "$RETRY_SLEEP_SECONDS"
    attempt=$((attempt + 1))
    # If the first attempt generated raw params before failing, reuse them.
    if [[ -f "$raw_dir/flat_parameters.pth" ]]; then
      raw_flag=0
    fi
  done
}

run_fq_worker_with_retry \
  llama31_fq_bc \
  "meta-llama/Meta-Llama-3.1-8B" \
  "$RESULTS_EVAL_DIR/flatquant_bc_Meta-Llama-3.1-8B_b3_full.json"

run_fq_worker_with_retry \
  llama3_fq_bc \
  "meta-llama/Meta-Llama-3-8B" \
  "$RESULTS_EVAL_DIR/flatquant_bc_Meta-Llama-3-8B_b3_full.json"

echo "Resume finished."


In [ ]:
%%bash
mkdir -p smart-flip/scripts/bash
rm -f smart-flip/scripts/bash/egbc_c4_b3_sequential.sh


In [ ]:
%%writefile smart-flip/scripts/bash/egbc_c4_b3_sequential.sh
#!/usr/bin/env bash
set -euo pipefail

# Sequential runner for cheap single-GPU instances.
# It reuses egbc_c4_b3.sh workers but runs one job at a time.

SCRIPT_DIR="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd)"
RUNNER="$SCRIPT_DIR/egbc_c4_b3.sh"

GPU="${GPU:-0}"
GPU_LIST="${GPU_LIST:-$GPU,$GPU,$GPU,$GPU}"
PYTHON_BIN="${PYTHON_BIN:-python}"
WANDB_PROJECT="${WANDB_PROJECT:-egbc_c4_b3}"
RESULTS_MODELS_DIR="${RESULTS_MODELS_DIR:-./results/models}"
HF_DATASETS_CACHE="${HF_DATASETS_CACHE:-./data/cache/hf_datasets/datasets-egbc-c4-b3}"

RUN_EGBC_BEST="${RUN_EGBC_BEST:-0}"
RUN_EGBC_SELECTED="${RUN_EGBC_SELECTED:-1}"
RUN_BC_BASELINE="${RUN_BC_BASELINE:-0}"
RUN_BC_EXTRA="${RUN_BC_EXTRA:-1}"
RUN_FLOAT_MODEL="${RUN_FLOAT_MODEL:-0}"

# Set RUN_FQ_RAW_ONCE=1 on fresh instances. Each FlatQuant worker will generate
# raw parameters only when its artifact is missing under RESULTS_MODELS_DIR.
RUN_FQ_RAW_ONCE="${RUN_FQ_RAW_ONCE:-1}"

run_one() {
  local worker="$1"
  shift

  echo "================================================================"
  echo "Starting sequential worker: $worker"
  echo "GPU=$GPU GPU_LIST=$GPU_LIST WANDB_PROJECT=$WANDB_PROJECT"
  echo "================================================================"

  env \
    GPU_LIST="$GPU_LIST" \
    PYTHON_BIN="$PYTHON_BIN" \
    WANDB_PROJECT="$WANDB_PROJECT" \
    RESULTS_MODELS_DIR="$RESULTS_MODELS_DIR" \
    HF_DATASETS_CACHE="$HF_DATASETS_CACHE" \
    HF_DATASETS_OFFLINE="${HF_DATASETS_OFFLINE:-0}" \
    WORKER="$worker" \
    "$@" \
    bash "$RUNNER"
}

flatquant_raw_dir() {
  local model_path="$1"
  local model_slug="${model_path##*/}"
  printf '%s/flatquant_raw/flatquant_raw_%s' "$RESULTS_MODELS_DIR" "$model_slug"
}

run_fq_one() {
  local worker="$1"
  local model_path="$2"
  local raw_flag=0
  local raw_dir
  raw_dir="$(flatquant_raw_dir "$model_path")"

  if [[ ! -f "$raw_dir/flat_parameters.pth" && "$RUN_FQ_RAW_ONCE" == "1" ]]; then
    raw_flag=1
  fi

  run_one "$worker" RUN_FQ_RAW="$raw_flag"
}

if [[ "$RUN_FLOAT_MODEL" == "1" ]]; then
  run_one qwen_float
  run_one llama31_float
  run_one llama3_float
fi

if [[ "$RUN_EGBC_BEST" == "1" ]]; then
  run_one qwen_awq_egbc_c4
  run_fq_one qwen_fq_egbc_c4 "Qwen/Qwen2.5-7B"
  run_one llama31_awq_egbc_c4
  run_one llama3_awq_egbc_c4
fi

if [[ "$RUN_EGBC_SELECTED" == "1" ]]; then
  run_one mistral_awq_selected_egbc_c4
  run_one llama31_awq_selected_egbc_c4
  run_one qwen_awq_selected_egbc_c4
  run_one llama3_awq_selected_egbc_c4

  run_fq_one mistral_fq_selected_egbc_c4 "mistralai/Mistral-7B-v0.3"
  run_fq_one llama31_fq_selected_egbc_c4 "meta-llama/Meta-Llama-3.1-8B"
  run_fq_one qwen_fq_selected_egbc_c4 "Qwen/Qwen2.5-7B"
  run_fq_one llama3_fq_selected_egbc_c4 "meta-llama/Meta-Llama-3-8B"
fi

if [[ "$RUN_BC_BASELINE" == "1" ]]; then
  run_one qwen_awq_bc
  run_fq_one qwen_fq_bc "Qwen/Qwen2.5-7B"
  run_one llama31_awq_bc
  run_one llama3_awq_bc
fi

if [[ "$RUN_BC_EXTRA" == "1" ]]; then
  run_one mistral_awq_bc
  run_fq_one mistral_fq_bc "mistralai/Mistral-7B-v0.3"
  run_fq_one llama31_fq_bc "meta-llama/Meta-Llama-3.1-8B"
  run_fq_one llama3_fq_bc "meta-llama/Meta-Llama-3-8B"
fi

echo "Sequential run finished."


In [ ]:
%%bash
mkdir -p smart-flip/scripts/bash
rm -f smart-flip/scripts/bash/egbc_c4_b3.sh


In [ ]:
%%writefile smart-flip/scripts/bash/egbc_c4_b3.sh
#!/usr/bin/env bash
set -euo pipefail

# Run 3-bit EGBC C4 checks and 3-bit +BC baselines.
# RUN_EGBC_BEST uses exact tuned configs from tune_best.
# RUN_EGBC_SELECTED uses untuned selected/default configs from the selected-run scripts.
# +BC intentionally uses method defaults; it is not tuned with Smart-Flip params.

GPU_LIST="${GPU_LIST:-0,1,2,3}"
IFS=',' read -ra _GPUS <<< "$GPU_LIST"
GPU_QWEN_AWQ="${_GPUS[0]:-0}"
GPU_QWEN_FQ="${_GPUS[1]:-1}"
GPU_LLAMA31_AWQ="${_GPUS[2]:-2}"
GPU_LLAMA3_AWQ="${_GPUS[3]:-3}"
GPU_MISTRAL_AWQ="${_GPUS[4]:-${_GPUS[0]:-0}}"
GPU_MISTRAL_FQ="${_GPUS[5]:-${_GPUS[1]:-1}}"
GPU_LLAMA31_FQ="${_GPUS[6]:-${_GPUS[2]:-2}}"
GPU_LLAMA3_FQ="${_GPUS[7]:-${_GPUS[3]:-3}}"

PYTHON_BIN="${PYTHON_BIN:-python}"
MODELS_ROOT="${MODELS_ROOT:-/models}"
RESULTS_MODELS_DIR="${RESULTS_MODELS_DIR:-./results/models}"
RESULTS_EVAL_DIR="${RESULTS_EVAL_DIR:-./results/eval}"
CALIBRATION_CACHE_DIR="${CALIBRATION_CACHE_DIR:-./data/cache/calibration}"
EVAL_CACHE_DIR="${EVAL_CACHE_DIR:-./data/cache/eval}"
HF_DATASETS_CACHE="${HF_DATASETS_CACHE:-./data/cache/hf_datasets/datasets-egbc-c4-b3}"
LOG_DIR="${LOG_DIR:-./logs/egbc_c4_b3}"

WANDB_PROJECT="${WANDB_PROJECT:-egbc_c4_b3}"
WANDB_ENTITY="${WANDB_ENTITY:-}"

BITS="3"
N_CALIB="${N_CALIB:-128}"
CALIB_SEQLEN="${CALIB_SEQLEN:-2048}"
CALIB_DATASET="${CALIB_DATASET:-c4}"
SEED="${SEED:-42}"
STRIDE="${STRIDE:-512}"
MAX_LENGTH="${MAX_LENGTH:-2048}"
C4_SAMPLES="${C4_SAMPLES:-500}"
BIAS_CORRECTION_SAMPLES="${BIAS_CORRECTION_SAMPLES:-4096}"
LM_EVAL_TASKS=(arc_challenge arc_easy boolq piqa rte)

RUN_EGBC_BEST="${RUN_EGBC_BEST:-0}"
RUN_EGBC_SELECTED="${RUN_EGBC_SELECTED:-1}"
RUN_BC_BASELINE="${RUN_BC_BASELINE:-0}"
RUN_BC_EXTRA="${RUN_BC_EXTRA:-1}"
RUN_FLOAT_MODEL="${RUN_FLOAT_MODEL:-0}"
RUN_FQ_RAW="${RUN_FQ_RAW:-0}"

FLATQUANT_EPOCHS="${FLATQUANT_EPOCHS:-15}"
FLATQUANT_CALI_BSZ="${FLATQUANT_CALI_BSZ:-4}"
FLATQUANT_LR="${FLATQUANT_LR:-5e-3}"
FLATQUANT_DIAG_INIT="${FLATQUANT_DIAG_INIT:-sq_style}"
FLATQUANT_DIAG_ALPHA="${FLATQUANT_DIAG_ALPHA:-0.3}"
FLATQUANT_CALI_TRANS="${FLATQUANT_CALI_TRANS:-1}"
FLATQUANT_ADD_DIAG="${FLATQUANT_ADD_DIAG:-1}"
FLATQUANT_LWC="${FLATQUANT_LWC:-1}"
FLATQUANT_LAC="${FLATQUANT_LAC:-1}"
FLATQUANT_ORTHOGONAL_MAP="${FLATQUANT_ORTHOGONAL_MAP:-matrix_exp}"
FLATQUANT_USE_TRIVIALIZATION="${FLATQUANT_USE_TRIVIALIZATION:-1}"

export FLATQUANT_ORTHOGONAL_MAP
export FLATQUANT_USE_TRIVIALIZATION
export HF_DATASETS_CACHE
export HF_DATASETS_OFFLINE="${HF_DATASETS_OFFLINE:-0}"

mkdir -p "$LOG_DIR" "$HF_DATASETS_CACHE"

wandb_args=()
if [[ -n "$WANDB_ENTITY" ]]; then
  wandb_args+=(--wandb-entity "$WANDB_ENTITY")
fi

add_flatquant_args() {
  local -n args_ref=$1
  args_ref+=(
    --flatquant-epochs "$FLATQUANT_EPOCHS"
    --flatquant-cali-bsz "$FLATQUANT_CALI_BSZ"
    --flatquant-lr "$FLATQUANT_LR"
    --flatquant-diag-init "$FLATQUANT_DIAG_INIT"
    --flatquant-diag-alpha "$FLATQUANT_DIAG_ALPHA"
  )
  [[ "$FLATQUANT_CALI_TRANS" == "1" ]] && args_ref+=(--flatquant-cali-trans) || args_ref+=(--no-flatquant-cali-trans)
  [[ "$FLATQUANT_ADD_DIAG" == "1" ]] && args_ref+=(--flatquant-add-diag) || args_ref+=(--no-flatquant-add-diag)
  [[ "$FLATQUANT_LWC" == "1" ]] && args_ref+=(--flatquant-lwc) || args_ref+=(--no-flatquant-lwc)
  [[ "$FLATQUANT_LAC" == "1" ]] && args_ref+=(--flatquant-lac) || args_ref+=(--no-flatquant-lac)
}

common_quant_args() {
  local model_path="$1"
  printf '%s\0' \
    --model-path "$model_path" \
    --models-root "$MODELS_ROOT" \
    --results-models-dir "$RESULTS_MODELS_DIR" \
    --results-eval-dir "$RESULTS_EVAL_DIR" \
    --calibration-cache-dir "$CALIBRATION_CACHE_DIR" \
    --eval-cache-dir "$EVAL_CACHE_DIR" \
    --calib-dataset "$CALIB_DATASET" \
    --n-calib "$N_CALIB" \
    --calib-seqlen "$CALIB_SEQLEN" \
    --seed "$SEED" \
    --stride "$STRIDE" \
    --max-length "$MAX_LENGTH" \
    --c4-samples "$C4_SAMPLES" \
    --bits "$BITS" \
    --use-wandb \
    --wandb-project "$WANDB_PROJECT" \
    "${wandb_args[@]}"
}

read_common_args() {
  local model_path="$1"
  local -n out_ref=$2
  mapfile -d '' -t out_ref < <(common_quant_args "$model_path")
}

flatquant_raw_dir() {
  local model_path="$1"
  local model_slug="${model_path##*/}"
  printf '%s/flatquant_raw/flatquant_raw_%s' "$RESULTS_MODELS_DIR" "$model_slug"
}

maybe_run_flatquant_raw() {
  local model_path="$1"
  local gpu="$2"
  local raw_dir
  raw_dir="$(flatquant_raw_dir "$model_path")"

  if [[ "$RUN_FQ_RAW" != "1" ]]; then
    if [[ ! -f "$raw_dir/flat_parameters.pth" ]]; then
      echo "Missing FlatQuant raw artifact: $raw_dir/flat_parameters.pth" >&2
      echo "Set RUN_FQ_RAW=1 to regenerate it, or set RESULTS_MODELS_DIR to the tune output root." >&2
      return 1
    fi
    return 0
  fi

  local args=()
  read_common_args "$model_path" args
  add_flatquant_args args

  CUDA_VISIBLE_DEVICES="$gpu" "$PYTHON_BIN" main.py quantize \
    "${args[@]}" \
    --origin-method flatquant \
    --post-correction none \
    --no-lm-eval \
    --run-name "flatquant_raw_${model_path##*/}"
}

run_egbc_awq_c4() {
  local model_path="$1"
  local gpu="$2"
  local knee="$3"
  local flip="$4"
  local model_slug="${model_path##*/}"
  local args=()
  read_common_args "$model_path" args

  CUDA_VISIBLE_DEVICES="$gpu" "$PYTHON_BIN" main.py quantize \
    "${args[@]}" \
    --origin-method awq \
    --post-correction smart_flip \
    --knee-tolerance "$knee" \
    --max-flip-percent "$flip" \
    --no-lm-eval \
    --run-name "c4_awq_sf_${model_slug}_b${BITS}_k${knee}_f${flip}"
}

run_egbc_flatquant_c4() {
  local model_path="$1"
  local gpu="$2"
  local knee="$3"
  local flip="$4"
  local model_slug="${model_path##*/}"
  local raw_dir
  raw_dir="$(flatquant_raw_dir "$model_path")"
  maybe_run_flatquant_raw "$model_path" "$gpu"

  local args=()
  read_common_args "$model_path" args
  add_flatquant_args args

  CUDA_VISIBLE_DEVICES="$gpu" "$PYTHON_BIN" main.py quantize \
    "${args[@]}" \
    --origin-method flatquant \
    --post-correction smart_flip \
    --knee-tolerance "$knee" \
    --max-flip-percent "$flip" \
    --flatquant-raw-path "$raw_dir" \
    --no-lm-eval \
    --run-name "c4_fq_sf_${model_slug}_b${BITS}_k${knee}_f${flip}"
}

run_bc_awq_full() {
  local model_path="$1"
  local gpu="$2"
  local model_slug="${model_path##*/}"
  local args=()
  read_common_args "$model_path" args

  CUDA_VISIBLE_DEVICES="$gpu" "$PYTHON_BIN" main.py quantize \
    "${args[@]}" \
    --origin-method awq \
    --post-correction bias_correction \
    --bias-correction-samples "$BIAS_CORRECTION_SAMPLES" \
    --lm-eval-tasks "${LM_EVAL_TASKS[@]}" \
    --run-name "awq_bc_${model_slug}_b${BITS}_full"
}

run_bc_flatquant_full() {
  local model_path="$1"
  local gpu="$2"
  local model_slug="${model_path##*/}"
  local raw_dir
  raw_dir="$(flatquant_raw_dir "$model_path")"
  maybe_run_flatquant_raw "$model_path" "$gpu"

  local args=()
  read_common_args "$model_path" args
  add_flatquant_args args

  CUDA_VISIBLE_DEVICES="$gpu" "$PYTHON_BIN" main.py quantize \
    "${args[@]}" \
    --origin-method flatquant \
    --post-correction bias_correction \
    --bias-correction-samples "$BIAS_CORRECTION_SAMPLES" \
    --flatquant-raw-path "$raw_dir" \
    --lm-eval-tasks "${LM_EVAL_TASKS[@]}" \
    --run-name "flatquant_bc_${model_slug}_b${BITS}_full"
}

run_float_full() {
  local model_path="$1"
  local gpu="$2"
  local model_slug="${model_path##*/}"

  CUDA_VISIBLE_DEVICES="$gpu" "$PYTHON_BIN" main.py float_model \
    --model-path "$model_path" \
    --models-root "$MODELS_ROOT" \
    --results-eval-dir "$RESULTS_EVAL_DIR" \
    --eval-cache-dir "$EVAL_CACHE_DIR" \
    --seed "$SEED" \
    --stride "$STRIDE" \
    --max-length "$MAX_LENGTH" \
    --c4-samples "$C4_SAMPLES" \
    --lm-eval-tasks "${LM_EVAL_TASKS[@]}" \
    --use-wandb \
    --wandb-project "$WANDB_PROJECT" \
    "${wandb_args[@]}" \
    --run-name "float_${model_slug}_full"
}

run_worker() {
  case "$1" in
    # Exact tuned best configs extracted from tune_best / tuning CSV.
    qwen_awq_egbc_c4) run_egbc_awq_c4 "Qwen/Qwen2.5-7B" "$GPU_QWEN_AWQ" "0.02" "0.03" ;;
    qwen_fq_egbc_c4) run_egbc_flatquant_c4 "Qwen/Qwen2.5-7B" "$GPU_QWEN_FQ" "0.02" "0.04" ;;
    llama31_awq_egbc_c4) run_egbc_awq_c4 "meta-llama/Meta-Llama-3.1-8B" "$GPU_LLAMA31_AWQ" "0.05" "0.03" ;;
    llama3_awq_egbc_c4) run_egbc_awq_c4 "meta-llama/Meta-Llama-3-8B" "$GPU_LLAMA3_AWQ" "0" "0.01" ;;

    # Untuned selected/default configs from run_awq_selected_egbc_b3.sh.
    mistral_awq_selected_egbc_c4) run_egbc_awq_c4 "mistralai/Mistral-7B-v0.3" "$GPU_MISTRAL_AWQ" "0.02" "0.02" ;;
    llama31_awq_selected_egbc_c4) run_egbc_awq_c4 "meta-llama/Meta-Llama-3.1-8B" "$GPU_LLAMA31_AWQ" "0.02" "0.01" ;;
    qwen_awq_selected_egbc_c4) run_egbc_awq_c4 "Qwen/Qwen2.5-7B" "$GPU_QWEN_AWQ" "0.03" "0.02" ;;
    llama3_awq_selected_egbc_c4) run_egbc_awq_c4 "meta-llama/Meta-Llama-3-8B" "$GPU_LLAMA3_AWQ" "0.01" "0.05" ;;

    # Untuned selected/default configs from run_flatquant_selected_egbc_b3.sh
    # and run_flatquant_llama3_egbc_b3.sh.
    mistral_fq_selected_egbc_c4) run_egbc_flatquant_c4 "mistralai/Mistral-7B-v0.3" "$GPU_MISTRAL_FQ" "0.0" "0.05" ;;
    llama31_fq_selected_egbc_c4) run_egbc_flatquant_c4 "meta-llama/Meta-Llama-3.1-8B" "$GPU_LLAMA31_FQ" "0.0" "0.05" ;;
    qwen_fq_selected_egbc_c4) run_egbc_flatquant_c4 "Qwen/Qwen2.5-7B" "$GPU_QWEN_FQ" "0.01" "0.05" ;;
    llama3_fq_selected_egbc_c4) run_egbc_flatquant_c4 "meta-llama/Meta-Llama-3-8B" "$GPU_LLAMA3_FQ" "0.02" "0.05" ;;

    # +BC baselines: default method parameters, full eval = 5 challenges + WikiText-2 + C4.
    qwen_awq_bc) run_bc_awq_full "Qwen/Qwen2.5-7B" "$GPU_QWEN_AWQ" ;;
    qwen_fq_bc) run_bc_flatquant_full "Qwen/Qwen2.5-7B" "$GPU_QWEN_FQ" ;;
    llama31_awq_bc) run_bc_awq_full "meta-llama/Meta-Llama-3.1-8B" "$GPU_LLAMA31_AWQ" ;;
    llama3_awq_bc) run_bc_awq_full "meta-llama/Meta-Llama-3-8B" "$GPU_LLAMA3_AWQ" ;;
    mistral_awq_bc) run_bc_awq_full "mistralai/Mistral-7B-v0.3" "$GPU_MISTRAL_AWQ" ;;
    mistral_fq_bc) run_bc_flatquant_full "mistralai/Mistral-7B-v0.3" "$GPU_MISTRAL_FQ" ;;
    llama31_fq_bc) run_bc_flatquant_full "meta-llama/Meta-Llama-3.1-8B" "$GPU_LLAMA31_FQ" ;;
    llama3_fq_bc) run_bc_flatquant_full "meta-llama/Meta-Llama-3-8B" "$GPU_LLAMA3_FQ" ;;
    qwen_float) run_float_full "Qwen/Qwen2.5-7B" "$GPU_QWEN_AWQ" ;;
    llama31_float) run_float_full "meta-llama/Meta-Llama-3.1-8B" "$GPU_LLAMA31_AWQ" ;;
    llama3_float) run_float_full "meta-llama/Meta-Llama-3-8B" "$GPU_LLAMA3_AWQ" ;;
    *) echo "Unknown WORKER: $1" >&2; return 2 ;;
  esac
}

run_phase() {
  local phase_name="$1"
  shift
  local status=0
  local pids=()
  local names=()

  echo "==> phase: $phase_name"
  for worker in "$@"; do
    echo "  starting $worker"
    run_worker "$worker" > "$LOG_DIR/${worker}.log" 2>&1 &
    pids+=("$!")
    names+=("$worker")
  done

  for i in "${!pids[@]}"; do
    if ! wait "${pids[$i]}"; then
      echo "FAILED: ${names[$i]} (log: $LOG_DIR/${names[$i]}.log)" >&2
      status=1
    fi
  done
  return "$status"
}

if [[ -n "${WORKER:-}" ]]; then
  run_worker "$WORKER" 2>&1 | tee "$LOG_DIR/${WORKER}.log"
  exit "${PIPESTATUS[0]}"
fi

echo "================================================================"
echo "egbc_c4_b3: project=$WANDB_PROJECT bits=$BITS c4_samples=$C4_SAMPLES"
echo "phases: best=$RUN_EGBC_BEST selected=$RUN_EGBC_SELECTED bc_baseline=$RUN_BC_BASELINE bc_extra=$RUN_BC_EXTRA float=$RUN_FLOAT_MODEL"
echo "logs: $LOG_DIR"
echo "================================================================"

if [[ "$RUN_FLOAT_MODEL" == "1" ]]; then
  run_phase float qwen_float llama31_float llama3_float
fi

if [[ "$RUN_EGBC_BEST" == "1" ]]; then
  run_phase egbc_best_c4 \
    qwen_awq_egbc_c4 \
    qwen_fq_egbc_c4 \
    llama31_awq_egbc_c4 \
    llama3_awq_egbc_c4
fi

if [[ "$RUN_EGBC_SELECTED" == "1" ]]; then
  run_phase egbc_selected_c4 \
    mistral_awq_selected_egbc_c4 \
    llama31_awq_selected_egbc_c4 \
    qwen_awq_selected_egbc_c4 \
    llama3_awq_selected_egbc_c4 \
    mistral_fq_selected_egbc_c4 \
    llama31_fq_selected_egbc_c4 \
    qwen_fq_selected_egbc_c4 \
    llama3_fq_selected_egbc_c4
fi

if [[ "$RUN_BC_BASELINE" == "1" ]]; then
  run_phase bc_baseline_full \
    qwen_awq_bc \
    qwen_fq_bc \
    llama31_awq_bc \
    llama3_awq_bc
fi

if [[ "$RUN_BC_EXTRA" == "1" ]]; then
  run_phase bc_extra_full \
    mistral_awq_bc \
    mistral_fq_bc \
    llama31_fq_bc \
    llama3_fq_bc
fi

echo "Done. Monitor or inspect logs in $LOG_DIR"


In [ ]:
%%bash
mkdir -p smart-flip/src/evaluation
rm -f smart-flip/src/evaluation/flatquant_data_utils.py


In [ ]:
%%writefile smart-flip/src/evaluation/flatquant_data_utils.py
import os
import pickle
import datasets
import random
import transformers


C4_TRAIN_URL = "https://huggingface.co/datasets/allenai/c4/resolve/main/en/c4-train.00000-of-01024.json.gz"
C4_VALIDATION_URL = "https://huggingface.co/datasets/allenai/c4/resolve/main/en/c4-validation.00000-of-00008.json.gz"


class TokenizerWrapper:
    def __init__(self, input_ids):
        self.input_ids = input_ids


def get_wikitext2(nsamples, seqlen, tokenizer, eval_mode=False):
    if eval_mode:
        testdata = datasets.load_dataset('./datasets/wikitext', 'wikitext-2-raw-v1', split='test')
        testenc = tokenizer("\n\n".join(testdata['text']), return_tensors='pt')
        return testenc
    else:
        traindata = datasets.load_dataset('./datasets/wikitext', 'wikitext-2-raw-v1', split='train')
        traindata = traindata.filter(lambda x: len(x) > 0)
        traindata = traindata.map(lambda x : {'text': x['text'].strip()})
        trainenc = tokenizer("\n\n".join(traindata['text']), return_tensors='pt')    
        trainloader = []
        for _ in range(nsamples):
            i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
            j = i + seqlen
            inp = trainenc.input_ids[:, i:j]
            tar = inp.clone()
            tar[:, :-1] = -100
            trainloader.append((inp, tar))
        return trainloader


def get_c4_new(nsamples, seqlen, tokenizer, eval_mode=False):
    split = "validation" if eval_mode else "train"
    url = C4_VALIDATION_URL if eval_mode else C4_TRAIN_URL
    dataset = datasets.load_dataset(
        "json",
        data_files={split: url},
        split=split,
        streaming=True,
    )

    if eval_mode:
        texts = []
        for item in dataset:
            text = item.get("text", "").strip()
            if not text:
                continue
            texts.append(text)
            if len(texts) >= 1100:
                break
        valenc = tokenizer(' '.join(texts), return_tensors='pt')
        valenc = valenc.input_ids[:, :(256 * seqlen)]
        valenc = TokenizerWrapper(valenc)
        return valenc

    trainloader = []
    for item in dataset:
        text = item.get("text", "").strip()
        if not text:
            continue
        trainenc = tokenizer(text, return_tensors='pt')
        if trainenc.input_ids.shape[1] < seqlen:
            continue
        max_start = trainenc.input_ids.shape[1] - seqlen
        start = random.randint(0, max_start)
        end = start + seqlen
        inp = trainenc.input_ids[:, start:end]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
        if len(trainloader) >= nsamples:
            break

    if len(trainloader) < nsamples:
        raise RuntimeError(f"Unable to collect {nsamples} C4 samples; only found {len(trainloader)} usable documents")
    return trainloader


def get_ptb_new(nsamples, seqlen, tokenizer, eval_mode=False):
    if eval_mode:
        testdata = datasets.load_dataset('./datasets/ptb_text_only', 'penn_treebank', split='test')
        testenc = tokenizer(" ".join(testdata['sentence']), return_tensors='pt')
        return testenc
    else:
        traindata = datasets.load_dataset('./datasets/ptb_text_only', 'penn_treebank', split='train')
        trainenc = tokenizer(" ".join(traindata['sentence']), return_tensors='pt')
        trainloader = []
        for _ in range(nsamples):
            i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
            j = i + seqlen
            inp = trainenc.input_ids[:, i:j]
            tar = inp.clone()
            tar[:, :-1] = -100
            trainloader.append((inp, tar))
        return trainloader


def get_pile(nsamples, seqlen, tokenizer):
    traindata = datasets.load_dataset("./datasets/pile-val-backup", split="validation")
    trainenc = tokenizer("\n\n".join(traindata['text'][:1000]), return_tensors='pt')
    trainloader = []
    for _ in range(nsamples):
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
    return trainloader


def get_loaders(
    args, name, tokenizer, nsamples=128, seqlen=2048, eval_mode=False
):
    if 'wikitext2' in name:
        dataset = get_wikitext2(nsamples, seqlen, tokenizer, eval_mode)
    elif 'ptb' in name:
        dataset = get_ptb_new(nsamples, seqlen, tokenizer, eval_mode)
    elif 'c4' in name:
        dataset = get_c4_new(nsamples, seqlen, tokenizer, eval_mode)
    elif 'pile' in name:
        dataset = get_pile(nsamples, seqlen, tokenizer)

    if 'c4' in name and eval_mode:
        dataset = dataset.input_ids
        dataset = TokenizerWrapper(dataset)
    return dataset


In [ ]:
%%bash
mkdir -p smart-flip/src/evaluation
rm -f smart-flip/src/evaluation/lm_eval.py


In [ ]:
%%writefile smart-flip/src/evaluation/lm_eval.py
"""
lm-evaluation-harness integration for downstream benchmark evaluation.
"""

from __future__ import annotations

import json
import os
import shutil
from datetime import datetime
from pathlib import Path
from typing import Dict

from src.io_utils import dump_json


class LMEvalHarnessRunner:
    def __init__(
        self,
        tasks: list[str],
        device: str = "cuda",
        batch_size: str = "auto",
        num_fewshot: int | None = None,
        output_dir: str = "./results/eval/lm_eval",
        run_name: str | None = None,
        hf_token: str | None = None,
    ):
        self.tasks = tasks
        self.device = device
        self.batch_size = batch_size
        self.num_fewshot = num_fewshot
        self.output_dir = Path(output_dir)
        self.run_name = run_name or datetime.now().strftime("%Y%m%d-%H%M%S")
        self.hf_token = hf_token
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _augment_known_runtime_error(self, exc: RuntimeError) -> RuntimeError:
        message = str(exc)
        if "Dataset scripts are no longer supported" not in message:
            return exc

        dataset_script = None
        if "found " in message:
            dataset_script = message.rsplit("found ", 1)[-1].strip()

        script_hint = f" ({dataset_script})" if dataset_script else ""
        return RuntimeError(
            "lm-eval is running with an incompatible Hugging Face `datasets` version. "
            f"Hugging Face removed dataset loading scripts in `datasets>=4.0`, and one of the requested tasks still needs one{script_hint}. "
            "Install a compatible version such as `datasets==2.17.1` (or any `datasets<4`) and rerun. "
            "If you only need perplexity, rerun with `INCLUDE_LM_EVAL=0` or pass `--no-lm-eval`."
        )

    def _augment_known_unicode_error(self, exc: UnicodeDecodeError) -> RuntimeError:
        message = str(exc)
        if "invalid start byte" not in message:
            return RuntimeError(f"lm-eval failed while loading a Hugging Face dataset: {message}")

        return RuntimeError(
            "lm-eval hit a known Hugging Face `datasets` bug while probing a dataset from the Hub. "
            "This has been observed with newer `datasets` 2.17/2.18 releases on some tasks. "
            "For the isolated Llama env, recreate the env with `requirements_llama3.txt`, which pins "
            "`lm-eval==0.4.1`, `datasets==2.14.6`, and `pyarrow==20.0.0`, then clear the old cache and rerun. "
            "In practice, clear both `~/.cache/huggingface/datasets` and dataset repos under "
            "`~/.cache/huggingface/hub/datasets--*`. "
            "If you only need perplexity, rerun with `INCLUDE_LM_EVAL=0` or pass `--no-lm-eval`."
        )

    def _augment_known_type_error(self, exc: TypeError) -> TypeError:
        message = str(exc)
        if "must be called with a dataclass type or instance" not in message:
            return exc

        return TypeError(
            "lm-eval hit an incompatible Hugging Face datasets cache. "
            "This usually happens when the current environment uses an older `datasets` version, "
            "but the local HF dataset cache was created by a newer version. "
            "Use a version-isolated HF datasets cache or delete the stale task cache and rerun. "
            "This repo now defaults `HF_DATASETS_CACHE` to `./data/cache/hf_datasets/datasets-<version>`; "
            "if you still see this on an existing server, clear the old global cache under `~/.cache/huggingface/datasets` "
            "for the affected tasks, or rerun with `INCLUDE_LM_EVAL=0` if you only need perplexity."
        )

    def _augment_known_import_error(self, exc: ImportError) -> RuntimeError:
        message = str(exc)

        if isinstance(exc, ModuleNotFoundError) and getattr(exc, "name", None) == "lm_eval":
            return RuntimeError(
                "lm-eval is not installed. Install the `lm-eval` package or disable lm-eval with `--no-lm-eval`."
            )

        if "huggingface_hub.errors" in message:
            return RuntimeError(
                "lm-eval is installed, but one of its dependencies is incompatible: "
                "`peft` is importing `huggingface_hub.errors`, which is missing from the currently installed "
                "`huggingface-hub` version. For the Llama env, install a newer hub release such as "
                "`huggingface-hub==0.24.6` and rerun. "
                "If you only need perplexity, rerun with `INCLUDE_LM_EVAL=0` or pass `--no-lm-eval`."
            )

        return RuntimeError(
            "lm-eval could not be imported because one of its dependencies failed to import. "
            f"Original import error: {message}"
        )

    def _hf_datasets_cache_dir(self) -> Path | None:
        cache_dir = os.getenv("HF_DATASETS_CACHE")
        if not cache_dir:
            return None
        return Path(cache_dir)

    def _can_clear_hf_datasets_cache(self, cache_dir: Path) -> bool:
        normalized_parts = {part.lower() for part in cache_dir.parts}
        return cache_dir.name.startswith("datasets-") and "hf_datasets" in normalized_parts

    def _clear_hf_datasets_cache(self):
        cache_dir = self._hf_datasets_cache_dir()
        if cache_dir is None:
            return False
        if not self._can_clear_hf_datasets_cache(cache_dir):
            return False

        shutil.rmtree(cache_dir, ignore_errors=True)
        cache_dir.mkdir(parents=True, exist_ok=True)
        print(f"\nDetected incompatible HF datasets cache, cleared {cache_dir} and retrying lm-eval once...")
        return True

    @staticmethod
    def _is_missing_hf_cache_error(exc: ValueError) -> bool:
        message = str(exc)
        return "Couldn't find cache for" in message and "Available configs in the cache" in message

    def _augment_known_value_error(self, exc: ValueError) -> RuntimeError | ValueError:
        if not self._is_missing_hf_cache_error(exc):
            return exc

        return RuntimeError(
            "lm-eval failed because the Hugging Face datasets cache is incomplete. "
            f"Original error: {exc}. "
            "This is a dataset-cache problem, not a quantization/model problem. "
            "Clear the isolated HF datasets cache and rerun, or prefetch both ARC configs "
            "`ARC-Challenge` and `ARC-Easy` before launching the experiment. "
            "Also make sure `HF_DATASETS_OFFLINE` is not set to `1`."
        )

    def _run_simple_evaluate(self, evaluator, model_name: str, model_path):
        if isinstance(model_path, dict) and {"model", "tokenizer"}.issubset(model_path):
            from lm_eval.models.huggingface import HFLM

            model = model_path["model"]
            tokenizer = model_path["tokenizer"]
            if self.device == "cuda":
                model = model.to(self.device)
            eval_batch_size = 1 if self.batch_size == "auto" else self.batch_size
            hflm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=eval_batch_size)
            return evaluator.simple_evaluate(
                model=hflm,
                tasks=self.tasks,
                device=self.device,
                batch_size=eval_batch_size,
                num_fewshot=self.num_fewshot,
                log_samples=False,
            )

        return evaluator.simple_evaluate(
            model="hf",
            model_args=self._model_args(model_path),
            tasks=self.tasks,
            device=self.device,
            batch_size=self.batch_size,
            num_fewshot=self.num_fewshot,
            log_samples=False,
        )

    def _model_args(self, model_path: str) -> str:
        dtype = "float16" if self.device == "cuda" else "float32"
        model_args = f"pretrained={model_path},dtype={dtype},trust_remote_code=True"
        if self.hf_token:
            model_args += f",token={self.hf_token}"
        return model_args

    def _make_json_safe(self, value):
        if value is None or isinstance(value, (str, int, float, bool)):
            return value

        if isinstance(value, Path):
            return str(value)

        if callable(value):
            name = getattr(value, "__name__", value.__class__.__name__)
            return f"<callable {name}>"

        if isinstance(value, dict):
            return {str(key): self._make_json_safe(item) for key, item in value.items()}

        if isinstance(value, (list, tuple, set)):
            return [self._make_json_safe(item) for item in value]

        item_method = getattr(value, "item", None)
        if callable(item_method):
            try:
                return self._make_json_safe(item_method())
            except (TypeError, ValueError):
                pass

        return repr(value)

    def _summarize_results(self, payload: dict) -> dict:
        results = payload.get("results", {})
        summary = {}
        for task_name, metrics in results.items():
            task_summary = {}
            for metric_name, value in metrics.items():
                if isinstance(value, (int, float)):
                    task_summary[metric_name] = value
            summary[task_name] = task_summary
        return summary

    def _write_raw_results(self, model_name: str, payload: dict):
        output_path = self.output_dir / f"{self.run_name}_{model_name}.json"
        safe_payload = self._make_json_safe(payload)
        dump_json(output_path, safe_payload, indent=2)

    def evaluate_model(self, model_name: str, model_path) -> dict:
        try:
            from lm_eval import evaluator
        except ImportError as exc:
            raise self._augment_known_import_error(exc) from exc

        try:
            payload = self._run_simple_evaluate(evaluator, model_name, model_path)
        except RuntimeError as exc:
            raise self._augment_known_runtime_error(exc) from exc
        except ValueError as exc:
            if self._is_missing_hf_cache_error(exc) and self._clear_hf_datasets_cache():
                try:
                    payload = self._run_simple_evaluate(evaluator, model_name, model_path)
                except RuntimeError as retry_exc:
                    raise self._augment_known_runtime_error(retry_exc) from retry_exc
                except ValueError as retry_exc:
                    raise self._augment_known_value_error(retry_exc) from retry_exc
                except UnicodeDecodeError as retry_exc:
                    raise self._augment_known_unicode_error(retry_exc) from retry_exc
                except TypeError as retry_exc:
                    raise self._augment_known_type_error(retry_exc) from retry_exc
            else:
                raise self._augment_known_value_error(exc) from exc
        except UnicodeDecodeError as exc:
            raise self._augment_known_unicode_error(exc) from exc
        except TypeError as exc:
            if "must be called with a dataclass type or instance" in str(exc) and self._clear_hf_datasets_cache():
                try:
                    payload = self._run_simple_evaluate(evaluator, model_name, model_path)
                except RuntimeError as retry_exc:
                    raise self._augment_known_runtime_error(retry_exc) from retry_exc
                except UnicodeDecodeError as retry_exc:
                    raise self._augment_known_unicode_error(retry_exc) from retry_exc
                except TypeError as retry_exc:
                    raise self._augment_known_type_error(retry_exc) from retry_exc
            else:
                raise self._augment_known_type_error(exc) from exc
        safe_payload = self._make_json_safe(payload)
        self._write_raw_results(model_name, payload)
        return {
            "tasks": list(self.tasks),
            "summary": self._summarize_results(payload),
            "raw": safe_payload,
        }

    def run(self, model_paths: Dict[str, str]) -> dict:
        results = {}
        for model_name, model_path in model_paths.items():
            print(f"\nRunning lm-eval for {model_name} on {', ' .join(self.tasks)}...")
            results[model_name] = self.evaluate_model(model_name, model_path)
        return results


In [ ]:
%%bash
mkdir -p smart-flip
rm -f smart-flip/requirements.txt


In [ ]:
%%writefile smart-flip/requirements.txt
numpy
# torch
transformers==4.45.0
datasets==2.17.1
accelerate==0.34.2
peft==0.13.2
huggingface-hub==0.24.6
sentencepiece
tqdm
psutil
lm-eval==0.4.9.1
wandb
python-dotenv


In [ ]:
%%bash
chmod +x smart-flip/scripts/bash/egbc_c4_b3_resume_bc_extra_from_llama.sh
chmod +x smart-flip/scripts/bash/egbc_c4_b3_sequential.sh
chmod +x smart-flip/scripts/bash/egbc_c4_b3.sh
